# 03 — Transformación Silver: Limpieza, Resolución de Entidades y Calidad (PDAC)
Estandariza el grafo biomédico al inglés, resuelve alias clínicos/farmacológicos, valida la ontología, enruta errores a cuarentena y audita métricas Antes vs. Después.

**1. Imports y configuración de tablas Delta en Silver**
Importa funciones nativas de PySpark y define las tablas origen, limpias, de cuarentena y de métricas.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

SOURCE_TABLE = "bronze.pdac.raw_triplets"
CLEAN_TABLE = "silver.pdac.clean_triplets"
QUARANTINE_TABLE = "silver.pdac.quarantine_triplets"
METRICS_TABLE = "silver.pdac.quality_metrics"

MIN_CONFIDENCE = 0.70
MIN_ENTITY_LEN = 2
MAX_ENTITY_LEN = 80

**2. Catálogo controlado de tipos de nodo y predicados**
Define la ontología permitida (`VALID_NODE_TYPES`), la prioridad determinista para resolver conflictos y el mapeo de predicados en `UPPER_SNAKE_CASE`.

In [ ]:
VALID_NODE_TYPES = [
    "DISEASE",
    "DRUG",
    "REGIMEN",
    "MUTATION",
    "BIOMARKER",
    "GENE",
    "PATHWAY",
    "PROCEDURE",
    "STAGE",
    "SYMPTOM",
    "RISK_FACTOR",
    "ADVERSE_EVENT",
    "TRIAL",
]

# Prioridad determinista para desempatar entidades con más de un tipo semántico
NODE_TYPE_PRIORITY = {
    "DRUG": 1,
    "REGIMEN": 2,
    "GENE": 3,
    "MUTATION": 4,
    "BIOMARKER": 5,
    "PATHWAY": 6,
    "DISEASE": 7,
    "STAGE": 8,
    "PROCEDURE": 9,
    "TRIAL": 10,
    "RISK_FACTOR": 11,
    "SYMPTOM": 12,
    "ADVERSE_EVENT": 13,
}

# Mapeo normalizado de predicados (UPPER_SNAKE_CASE en inglés)
PREDICATE_MAP = {
    "TREATS": "TREATS",
    "TRATA": "TREATS",
    "COMBINES_WITH": "COMBINES_WITH",
    "CAUSES": "CAUSES",
    "CONTRAINDICATED_IN": "CONTRAINDICATED_FOR",
    "CONTRAINDICATED_FOR": "CONTRAINDICATED_FOR",
    "TARGETS": "TARGETS",
    "EVALUATED_IN": "EVALUATED_IN",
    "DIAGNOSES": "DIAGNOSES",
    "ASSOCIATED_WITH": "ASSOCIATED_WITH",
    "BIOMARKER_FOR": "BIOMARKER_FOR",
    "HAS_MUTATION": "HAS_MUTATION",
    "RISK_FACTOR_FOR": "RISK_FACTOR_FOR",
    "INHIBITS": "INHIBITS",
}

VALID_PREDICATES = set(PREDICATE_MAP.values())

**3. Diccionario de alias clínicos, farmacológicos y traducciones ES→EN**
Consolida sinónimos, principios activos, marcas comerciales, acrónimos y términos en español detectados en `data/03_processed/`.

In [ ]:
# Diccionarios exhaustivos de resolución de entidades y traducción ES -> EN
# Construidos a partir de la inspección real de los 22 JSON en data/03_processed/
DRUG_ALIASES = {
    "gemcitabina": "Gemcitabine",
    "gemcitabine": "Gemcitabine",
    "gem": "Gemcitabine",
    "gemcitabina hospira": "Gemcitabine",
    "paclitaxel": "Paclitaxel",
    "paclitaxel albumina": "Nab-Paclitaxel",
    "paclitaxel-albumina": "Nab-Paclitaxel",
    "paclitaxel unido a albumina": "Nab-Paclitaxel",
    "paclitaxel albumina dr. reddys": "Nab-Paclitaxel",
    "nab-paclitaxel": "Nab-Paclitaxel",
    "nab paclitaxel": "Nab-Paclitaxel",
    "albumin-bound paclitaxel": "Nab-Paclitaxel",
    "abraxane": "Nab-Paclitaxel",
    "oxaliplatino": "Oxaliplatin",
    "oxaliplatin": "Oxaliplatin",
    "irinotecan": "Irinotecan",
    "irinotecan liposomal": "Liposomal Irinotecan",
    "irinotecan nanoliposomal": "Liposomal Irinotecan",
    "nanoliposomal irinotecan": "Liposomal Irinotecan",
    "nanoliposomal irinotecane": "Liposomal Irinotecan",
    "liposomal irinotecan": "Liposomal Irinotecan",
    "(nanoliposomal) irinotecan": "Liposomal Irinotecan",
    "irinotecan liposome": "Liposomal Irinotecan",
    "nal-iri": "Liposomal Irinotecan",
    "5-fluorouracilo (5-fu)": "5-Fluorouracil",
    "5-fluorouracilo": "5-Fluorouracil",
    "5-fluorouracil": "5-Fluorouracil",
    "fluorouracilo": "5-Fluorouracil",
    "fluorouracil": "5-Fluorouracil",
    "5-fu": "5-Fluorouracil",
    "5fu": "5-Fluorouracil",
    "continuous infusion fluorouracil": "5-Fluorouracil",
    "bolus fluorouracil": "5-Fluorouracil",
    "capecitabina": "Capecitabine",
    "capecitabine": "Capecitabine",
    "cisplatino": "Cisplatin",
    "cisplatin": "Cisplatin",
    "carboplatino": "Carboplatin",
    "carboplatin": "Carboplatin",
    "leucovorina": "Leucovorin",
    "leucovorina simplificada": "Leucovorin",
    "leucovorin": "Leucovorin",
    "acido folinico": "Leucovorin",
    "folinic acid": "Leucovorin",
    "fa": "Leucovorin",
    "rmc-6236": "Daraxonrasib",
    "daraxonrasib": "Daraxonrasib",
    "mrtx1133": "MRTX1133",
    "kras g12d-directed mrtx1133 inhibitor": "MRTX1133",
    "ds-8201a": "Trastuzumab Deruxtecan",
    "ds-8201a (trastuzumab deruxtecan)": "Trastuzumab Deruxtecan",
    "fam-trastuzumab deruxtecan-nxki": "Trastuzumab Deruxtecan",
    "trastuzumab deruxtecan": "Trastuzumab Deruxtecan",
    "selumetinib": "Selumetinib",
    "selumetinib (azd6244 hydrogen sulfate)": "Selumetinib",
    "pembrolizumab": "Pembrolizumab",
    "nivolumab": "Nivolumab",
    "ipilimumab": "Ipilimumab",
    "sotorasib": "Sotorasib",
    "adagrasib": "Adagrasib",
    "olaparib": "Olaparib",
    "erlotinib": "Erlotinib",
    "ivosidenib": "Ivosidenib",
    "entrectinib": "Entrectinib",
    "larotrectinib": "Larotrectinib",
    "dabrafenib": "Dabrafenib",
    "trametinib": "Trametinib",
    "selpercatinib": "Selpercatinib",
    "dostarlimab-gxly": "Dostarlimab-gxly",
    "repotrectinib": "Repotrectinib",
    "erdafitinib": "Erdafitinib",
    "zenocutuzumab-zbco": "Zenocutuzumab-zbco",
    "pemigatinib": "Pemigatinib",
    "nimotuzumab": "Nimotuzumab",
    "cetuximab": "Cetuximab",
    "bevacizumab": "Bevacizumab",
    "regorafenib": "Regorafenib",
    "crizotinib": "Crizotinib",
    "idelalisib": "Idelalisib",
    "sorafenib": "Sorafenib",
    "ruxolitinib": "Ruxolitinib",
    "metformin": "Metformin",
    "cyclophosphamide": "Cyclophosphamide",
    "docetaxel": "Docetaxel",
    "veliparib": "Veliparib",
    "pasireotide": "Pasireotide",
    "ciprofloxacin": "Ciprofloxacin",
    "leflunomide": "Leflunomide",
    "decitabine": "Decitabine",
    "atazanavir": "Atazanavir",
    "aldoxorubicin hcl": "Aldoxorubicin HCl",
    "adebrelimab (shr-a1921)": "Adebrelimab",
    "th-z835": "TH-Z835",
    "thz835": "TH-Z835",
    "cbp501": "CBP501",
    "cbp501 (25)": "CBP501",
    "cbp501 (16)": "CBP501",
    "epirubicina": "Epirubicin",
    "olanzapina": "Olanzapine",
    "minociclina": "Minocycline",
    "simvastatina": "Simvastatin",
    "warfarina": "Warfarin",
    "warfarin": "Warfarin",
    "ketoconazol": "Ketoconazole",
    "ciclosporina": "Cyclosporine",
    "tacrolimus": "Tacrolimus",
    "heparina de bajo peso molecular (hbpm)": "Low-Molecular-Weight Heparin",
    "low-molecular-weight heparin": "Low-Molecular-Weight Heparin",
    "prophylactic low-molecular-weight heparin": "Low-Molecular-Weight Heparin",
    "lmwh therapy": "Low-Molecular-Weight Heparin",
    "vacunas vivas atenuadas": "Live Attenuated Vaccines",
    "live vaccines": "Live Attenuated Vaccines",
    "vacunas contra la fiebre amarilla": "Yellow Fever Vaccine",
    "vacunas muertas o inactivadas": "Inactivated Vaccines",
    "intranasal influenza vaccines": "Intranasal Influenza Vaccines",
    "hierba de san juan": "St. John's Wort",
    "inhibidores o inductores potentes del citocromo p450 3a4": "CYP3A4 Inhibitors/Inducers",
    "inhibidores de la cinasa": "Kinase Inhibitors",
    "vitamina k antagonistas": "Vitamin K Antagonists",
    "agentes bloqueantes neuromusculares": "Neuromuscular Blocking Agents",
    "fluoropirimidina": "Fluoropyrimidines",
    "fluoropyrimidines": "Fluoropyrimidines",
    "ascorbato farmacologico intravenoso": "Intravenous Ascorbate",
    "ascorbato intravenoso": "Intravenous Ascorbate",
    "berahialuronidasa alfa-pmph": "Berahyaluronidase Alfa-pmph",
    "berahyaluronidase alfa-pmph": "Berahyaluronidase Alfa-pmph",
    "albumina": "Human Serum Albumin",
    "albumina serica humana": "Human Serum Albumin",
    "iodine-containing contrast agents": "Iodinated Contrast Agents",
    "intravenous iodinated non-ionic contrast agent": "Iodinated Contrast Agents",
    "s-1": "S-1",
    "n-803": "N-803",
    "pd-l1 t-hank": "PD-L1 t-haNK",
    "ast2169 liposome": "AST2169 Liposome",
    "hrs-4642": "HRS-4642",
    "bi-3406": "BI-3406",
    "bi-1701963": "BI-1701963",
    "rmc-9805": "RMC-9805",
    "kras g12d inhibitors": "KRAS G12D Inhibitors",
    "kras-targeting drugs": "KRAS Inhibitors",
    "kras inhibitors": "KRAS Inhibitors",
    "mek inhibitors": "MEK Inhibitors",
    "inhibitors of mek": "MEK Inhibitors",
    "inhibitors of pi3k": "PI3K Inhibitors",
    "inhibitors of cdk4/6": "CDK4/6 Inhibitors",
    "sos1 inhibitors": "SOS1 Inhibitors",
    "shp2 inhibitors": "SHP2 Inhibitors",
    "ldh inhibitors": "LDH Inhibitors",
    "anti-pd-1 checkpoint antibody": "Anti-PD-1 Antibody",
    "anti-pd-l1 monoclonal antibody": "Anti-PD-L1 Antibody",
    "alpha ctla-4": "Anti-CTLA-4 Antibody",
    "α ctla-4": "Anti-CTLA-4 Antibody",
    "α pd-1": "Anti-PD-1 Antibody",
    "α fasl": "Anti-FASL Antibody",
}

REGIMEN_ALIASES = {
    "folfirinox": "FOLFIRINOX",
    "folfirinox/mfolfirinox": "FOLFIRINOX",
    "mfolfirinox": "mFOLFIRINOX",
    "modified folfirinox": "mFOLFIRINOX",
    "folfirinox modificado": "mFOLFIRINOX",
    "neoadjuvant folfirinox": "Neoadjuvant FOLFIRINOX",
    "neoadjuvant mfolfirinox": "Neoadjuvant mFOLFIRINOX",
    "adjuvant folfirinox": "Adjuvant FOLFIRINOX",
    "nalirifox": "NALIRIFOX",
    "nanoliposomal irinotecan-5-fu-lv": "NALIRIFOX",
    "5-fluorouracil + leucovorin + nanoliposomal irinotecan": "NALIRIFOX",
    "gnp": "Gemcitabine + Nab-Paclitaxel",
    "gem-nabp": "Gemcitabine + Nab-Paclitaxel",
    "gn": "Gemcitabine + Nab-Paclitaxel",
    "gemcitabine-nab-paclitaxel (gn)": "Gemcitabine + Nab-Paclitaxel",
    "gemcitabine with albumin-bound paclitaxel": "Gemcitabine + Nab-Paclitaxel",
    "gemcitabine-albumin-bound paclitaxel": "Gemcitabine + Nab-Paclitaxel",
    "albumin-bound paclitaxel/gemcitabine": "Gemcitabine + Nab-Paclitaxel",
    "gemcitabina + paclitaxel": "Gemcitabine + Nab-Paclitaxel",
    "paclitaxel unido a albumina y gemcitabina": "Gemcitabine + Nab-Paclitaxel",
    "gemcitabine + albumin-bound paclitaxel": "Gemcitabine + Nab-Paclitaxel",
    "gemcitabine + nab-paclitaxel": "Gemcitabine + Nab-Paclitaxel",
    "5-fu/fa": "5-Fluorouracil/Leucovorin",
    "5fu/af": "5-Fluorouracil/Leucovorin",
    "5-fu/af": "5-Fluorouracil/Leucovorin",
    "5-fu-lv": "5-Fluorouracil/Leucovorin",
    "fluorouracil//leucovorin": "5-Fluorouracil/Leucovorin",
    "fluorouracil/leucovorin": "5-Fluorouracil/Leucovorin",
    "5-fluorouracil/acido folinico": "5-Fluorouracil/Leucovorin",
    "bolus fluorouracil/leucovorin": "5-Fluorouracil/Leucovorin",
    "gemcitabine-cisplatin": "Gemcitabine + Cisplatin",
    "cisplatin-gemcitabine": "Gemcitabine + Cisplatin",
    "gemcitabine-capecitabine": "Gemcitabine + Capecitabine",
    "capecitabina/gemcitabina": "Gemcitabine + Capecitabine",
    "gemcitabina + capecitabina": "Gemcitabine + Capecitabine",
    "gemcitabina + carboplatino": "Gemcitabine + Carboplatin",
    "dabrafenib-trametinib": "Dabrafenib + Trametinib",
    "ipilimumab + nivolumab": "Ipilimumab + Nivolumab",
    "nabplagem": "NABPLAGEM",
    "albumin-bound paclitaxel/cisplatin/gemcitabine": "NABPLAGEM",
    "folfox": "FOLFOX",
    "folfox4": "FOLFOX4",
    "mfolfox": "mFOLFOX",
    "mfolfox6": "mFOLFOX6",
    "folfiri": "FOLFIRI",
    "paxg": "PAXG",
    "capeox": "CAPEOX",
    "off": "OFF",
    "gtx": "GTX",
    "mvac": "MVAC",
    "firgem regimen": "FIRGEM",
    "quimioterapia adyuvante": "Adjuvant Chemotherapy",
    "adjuvant cht": "Adjuvant Chemotherapy",
    "adjuvant chemotherapy": "Adjuvant Chemotherapy",
    "adjuvant therapy": "Adjuvant Chemotherapy",
    "quimioterapia neoadyuvante": "Neoadjuvant Chemotherapy",
    "neoadjuvant cht": "Neoadjuvant Chemotherapy",
    "neoadjuvant chemotherapy": "Neoadjuvant Chemotherapy",
    "neoadyuvante": "Neoadjuvant Chemotherapy",
    "neoadjuvant therapy": "Neoadjuvant Chemotherapy",
    "induction cht": "Induction Chemotherapy",
    "tratamiento sistemico": "Systemic Chemotherapy",
    "systemic chemotherapy": "Systemic Chemotherapy",
    "systemic therapy": "Systemic Chemotherapy",
    "tratamiento sistemico de primera linea": "First-Line Systemic Chemotherapy",
    "first-line chemotherapy": "First-Line Systemic Chemotherapy",
    "second-line therapy": "Second-Line Chemotherapy",
    "second-line chemotherapy": "Second-Line Chemotherapy",
    "oxaliplatin-based second-line treatment": "Oxaliplatin-Based Second-Line Chemotherapy",
    "quimioterapia basada en gemcitabina": "Gemcitabine-Based Chemotherapy",
    "gemcitabine-based regimen": "Gemcitabine-Based Chemotherapy",
    "gemcitabine-based chemotherapy": "Gemcitabine-Based Chemotherapy",
    "gemcitabine- or paclitaxel-based therapy": "Gemcitabine- or Paclitaxel-Based Chemotherapy",
    "gemcitabine-based chemoradiation": "Gemcitabine-Based Chemoradiation",
    "gemcitabine-based chemoradiotherapy": "Gemcitabine-Based Chemoradiation",
    "gemcitabine-based neoadjuvant chemoradiation treatment": "Gemcitabine-Based Neoadjuvant Chemoradiation",
    "capecitabine-based chemoradiation": "Capecitabine-Based Chemoradiation",
    "fluorouracil-based chemoradiation": "5-Fluorouracil-Based Chemoradiation",
    "quimiorradioterapia": "Chemoradiation",
    "quimiorradiacion": "Chemoradiation",
    "chemoradiation": "Chemoradiation",
    "chemoradiotherapy": "Chemoradiation",
    "chemo(radio)therapy": "Chemoradiation",
    "crt": "Chemoradiation",
    "quimiorradiacion o rtce": "Chemoradiation / SBRT",
    "gnp -> srg": "Gemcitabine + Nab-Paclitaxel -> Surgery",
    "folfirinox -> srg": "FOLFIRINOX -> Surgery",
    "mfolfirinox -> srg": "mFOLFIRINOX -> Surgery",
    "paxg -> srg": "PAXG -> Surgery",
    "gem-s-1 -> srg": "Gemcitabine + S-1 -> Surgery",
    "ct -> crt -> srg": "Chemotherapy -> Chemoradiation -> Surgery",
    "ct -> ct -> srg": "Chemotherapy -> Chemotherapy -> Surgery",
    "cbp501/cisplatin/nivolumab combinations": "CBP501 + Cisplatin + Nivolumab",
}

DISEASE_ALIASES = {
    "pancreatic ductal adenocarcinoma": "Pancreatic Ductal Adenocarcinoma",
    "pdac": "Pancreatic Ductal Adenocarcinoma",
    "pda": "Pancreatic Ductal Adenocarcinoma",
    "pdac tumor": "Pancreatic Ductal Adenocarcinoma",
    "adenocarcinoma de pancreas": "Pancreatic Ductal Adenocarcinoma",
    "adenocarcinoma de pancreas ductal": "Pancreatic Ductal Adenocarcinoma",
    "carcinoma ductal pancreatico": "Pancreatic Ductal Adenocarcinoma",
    "pancreatic adenocarcinoma": "Pancreatic Ductal Adenocarcinoma",
    "pancreatic adenocarcinomas": "Pancreatic Ductal Adenocarcinoma",
    "ductal adenocarcinoma": "Pancreatic Ductal Adenocarcinoma",
    "pancreatic carcinoma": "Pancreatic Ductal Adenocarcinoma",
    "invasive pancreatic ductal adenocarcinoma": "Pancreatic Ductal Adenocarcinoma",
    "pancreatic cancer": "Pancreatic Cancer",
    "cancer pancreatico": "Pancreatic Cancer",
    "cancer pancreatica": "Pancreatic Cancer",
    "cancer de pancreas": "Pancreatic Cancer",
    "pc": "Pancreatic Cancer",
    "pancreatic tumors": "Pancreatic Cancer",
    "metastatic pancreatic ductal adenocarcinoma": "Metastatic Pancreatic Ductal Adenocarcinoma",
    "metastatic pdac": "Metastatic Pancreatic Ductal Adenocarcinoma",
    "mpdac": "Metastatic Pancreatic Ductal Adenocarcinoma",
    "cancer pancreatico metastasico": "Metastatic Pancreatic Ductal Adenocarcinoma",
    "cancer de pancreas metastasico": "Metastatic Pancreatic Ductal Adenocarcinoma",
    "metastatic pancreatic cancer": "Metastatic Pancreatic Cancer",
    "locally advanced pancreatic ductal adenocarcinoma": "Locally Advanced Pancreatic Ductal Adenocarcinoma",
    "lapc": "Locally Advanced Pancreatic Ductal Adenocarcinoma",
    "locally advanced pancreatic adenocarcinoma": "Locally Advanced Pancreatic Ductal Adenocarcinoma",
    "locally advanced pancreatic cancer": "Locally Advanced Pancreatic Cancer",
    "cancer de pancreas localmente avanzado": "Locally Advanced Pancreatic Cancer",
    "cancer de pancreas localmente avanzado irresecable": "Locally Advanced Unresectable Pancreatic Cancer",
    "locally advanced unresectable disease": "Locally Advanced Unresectable Pancreatic Cancer",
    "borderline resectable pancreatic ductal adenocarcinoma": "Borderline Resectable Pancreatic Ductal Adenocarcinoma",
    "cancer de pancreas de resecabilidad limitrofe": "Borderline Resectable Pancreatic Cancer",
    "cancer de pancreas resecable limitrofe": "Borderline Resectable Pancreatic Cancer",
    "brpc": "Borderline Resectable Pancreatic Cancer",
    "resectable pancreatic ductal adenocarcinoma": "Resectable Pancreatic Ductal Adenocarcinoma",
    "resectable pc": "Resectable Pancreatic Ductal Adenocarcinoma",
    "cancer de pancreas resecable": "Resectable Pancreatic Cancer",
    "resectable tumors": "Resectable Pancreatic Cancer",
    "cancer de pancreas irresecable": "Unresectable Pancreatic Cancer",
    "unresectable tumors": "Unresectable Pancreatic Cancer",
    "unresectable pancreatic ductal adenocarcinoma": "Unresectable Pancreatic Ductal Adenocarcinoma",
    "advanced pancreatic ductal adenocarcinoma": "Advanced Pancreatic Ductal Adenocarcinoma",
    "adenocarcinoma pancreatico avanzado": "Advanced Pancreatic Ductal Adenocarcinoma",
    "advanced pdac": "Advanced Pancreatic Ductal Adenocarcinoma",
    "advanced pancreatic cancer": "Advanced Pancreatic Cancer",
    "locally advanced or metastatic pancreatic cancer": "Locally Advanced or Metastatic Pancreatic Cancer",
    "resected pancreatic ductal adenocarcinoma": "Resected Pancreatic Ductal Adenocarcinoma",
    "cancer pancreatico metastasico con mutacion germinal en brca1 o brca2": "BRCA1/2-Mutated Metastatic Pancreatic Cancer",
    "brca1/2-associated metastatic pancreatic cancer": "BRCA1/2-Mutated Metastatic Pancreatic Cancer",
    "metastatic brca1/2 or palb2-associated metastatic pancreatic cancer": "BRCA1/2 or PALB2-Mutated Pancreatic Cancer",
    "brca1/2 or palb2 mutant pancreas cancer": "BRCA1/2 or PALB2-Mutated Pancreatic Cancer",
    "cancer pancreatico hereditario": "Hereditary Pancreatic Cancer",
    "hereditary pancreatic cancer (hpc)": "Hereditary Pancreatic Cancer",
    "familial pancreatic cancer (fpc)": "Familial Pancreatic Cancer",
    "familial pc": "Familial Pancreatic Cancer",
    "tumor en el cuello del pancreas": "Pancreatic Neck Tumor",
    "pancreas neck adenocarcinomas": "Pancreatic Neck Tumor",
    "tumor en el cuerpo o la cola del pancreas": "Pancreatic Body or Tail Tumor",
    "tumor del cuerpo o cola del pancreas": "Pancreatic Body or Tail Tumor",
    "tumores del cuerpo o cola del pancreas": "Pancreatic Body or Tail Tumor",
    "tumor en la cabeza del pancreas": "Pancreatic Head Tumor",
    "tumor de cabeza del pancreas": "Pancreatic Head Tumor",
    "tumores de cabeza del pancreas": "Pancreatic Head Tumor",
    "tumores del bazo": "Splenic Tumors",
    "btc": "Biliary Tract Cancer",
    "biliary tract cancer": "Biliary Tract Cancer",
    "biliary tract cancers": "Biliary Tract Cancer",
    "icc": "Intrahepatic Cholangiocarcinoma",
    "intrahepatic cholangiocarcinoma": "Intrahepatic Cholangiocarcinoma",
    "intrahepatic cholangiocarcinoma (icc)": "Intrahepatic Cholangiocarcinoma",
    "ecc": "Extrahepatic Cholangiocarcinoma",
    "extrahepatic cholangiocarcinoma (ecc)": "Extrahepatic Cholangiocarcinoma",
    "pecc": "Perihilar Cholangiocarcinoma",
    "perihilar cholangiocarcinoma": "Perihilar Cholangiocarcinoma",
    "decc": "Distal Cholangiocarcinoma",
    "distal cholangiocarcinoma": "Distal Cholangiocarcinoma",
    "liver flukes": "Liver Flukes",
    "recurrent disease": "Recurrent Disease",
    "crecimiento del cancer hacia los vasos sanguineos": "Vascular Invasion",
    "muerte por cancer": "Cancer-Related Mortality",
    "reduccion de la produccion de enzimas digestivas": "Reduced Digestive Enzyme Production",
    "reduccion de la produccion de insulina": "Reduced Insulin Production",
    "insuficiencia renal grave (aclaramiento de creatinina inferior a 30 ml/min)": "Severe Renal Impairment",
    "gbc": "Gallbladder Cancer",
    "gallbladder cancer": "Gallbladder Cancer",
    "gallbladder cancer (gbc)": "Gallbladder Cancer",
    "hcc": "Hepatocellular Carcinoma",
    "nsclc": "Non-Small Cell Lung Cancer",
    "non-small cell lung cancer": "Non-Small Cell Lung Cancer",
    "cancer de pulmon no microcitico": "Non-Small Cell Lung Cancer",
    "cancer de pulmon no microcitico (cpnm)": "Non-Small Cell Lung Cancer",
    "cancer de pulmon no microcitico avanzado": "Advanced Non-Small Cell Lung Cancer",
    "cancer de pulmon no microcitico en estadio iii": "Stage III Non-Small Cell Lung Cancer",
    "cancer de pulmon no microcitico en estadio iiib": "Stage IIIB Non-Small Cell Lung Cancer",
    "cancer de pulmon no microcitico en estadio iiib/iv": "Stage IIIB/IV Non-Small Cell Lung Cancer",
    "cancer de pulmon no microcitico en estadio iv": "Stage IV Non-Small Cell Lung Cancer",
    "cancer de pulmon no microcitico de histologia escamosa": "Squamous Non-Small Cell Lung Cancer",
    "cancer de pulmon no microcitico de histologia no escamosa": "Non-Squamous Non-Small Cell Lung Cancer",
    "cancer de pulmon": "Lung Cancer",
    "cancer de pulmones": "Lung Cancer",
    "lung cancer": "Lung Cancer",
    "cancer de mama": "Breast Cancer",
    "cancer de mama metastasico": "Metastatic Breast Cancer",
    "cancer de ovario": "Ovarian Cancer",
    "cancer de vejiga": "Bladder Cancer",
    "cancer de prostata": "Prostate Cancer",
    "cancer de higado": "Liver Cancer",
    "cancer de colon": "Colon Cancer",
    "colon cancer": "Colon Cancer",
    "cancer de colon de estadio iii (c de duke)": "Stage III Colon Cancer",
    "cancer colorrectal": "Colorectal Cancer",
    "colorectal cancer": "Colorectal Cancer",
    "cancer de colon colorrectal": "Colorectal Cancer",
    "cancer colorrectal avanzado": "Advanced Colorectal Cancer",
    "cancer colorrectal metastasico": "Metastatic Colorectal Cancer",
    "carcinoma metastasico de colon o recto": "Metastatic Colorectal Cancer",
    "carcinoma colorrectal metastasico": "Metastatic Colorectal Cancer",
    "cancer de colorrectal metastasico": "Metastatic Colorectal Cancer",
    "cancer de colon colorrectal metastasico": "Metastatic Colorectal Cancer",
    "cancer de colon colorrectal previamente tratado": "Previously Treated Colorectal Cancer",
    "cancer de revestimiento del abdomen": "Peritoneal Cancer",
    "sarcoma de ewing": "Ewing Sarcoma",
    "neuroblastoma": "Neuroblastoma",
    "rabdomiosarcoma": "Rhabdomyosarcoma",
    "tumores solidos": "Solid Tumors",
    "solid tumors": "Solid Tumors",
    "solid cancers": "Solid Tumors",
    "advanced solid tumors": "Advanced Solid Tumors",
    "tumores solidos recurrentes o resistentes": "Recurrent or Refractory Solid Tumors",
    "cancer metastasico": "Metastatic Cancer",
    "intraductal papillarymucinous neoplasms -ipmn": "Intraductal Papillary Mucinous Neoplasm (IPMN)",
    "intraductal papillary mucinous neoplasm": "Intraductal Papillary Mucinous Neoplasm (IPMN)",
    "intraductal papillary mucinous neoplasms (ipmns)": "Intraductal Papillary Mucinous Neoplasm (IPMN)",
    "ipmns": "Intraductal Papillary Mucinous Neoplasm (IPMN)",
    "pancreatic intraepithelial neoplasia-panin": "Pancreatic Intraepithelial Neoplasia (PanIN)",
    "pancreatic intraepithelial neoplasia": "Pancreatic Intraepithelial Neoplasia (PanIN)",
    "pancreatic intraepithelial neoplasia (panin)": "Pancreatic Intraepithelial Neoplasia (PanIN)",
    "panins": "Pancreatic Intraepithelial Neoplasia (PanIN)",
    "mucinous cystic neoplasm": "Mucinous Cystic Neoplasm (MCN)",
    "mucinous cystic neoplasms (mcns)": "Mucinous Cystic Neoplasm (MCN)",
    "mcns": "Mucinous Cystic Neoplasm (MCN)",
    "diabetes": "Diabetes Mellitus",
    "type 2 diabetes": "Type 2 Diabetes Mellitus",
    "type 2 diabetes mellitus": "Type 2 Diabetes Mellitus",
    "new onset diabetes": "New-Onset Diabetes Mellitus",
    "new-onset diabetes": "New-Onset Diabetes Mellitus",
    "diabetes de reciente aparicion": "New-Onset Diabetes Mellitus",
    "insuficiencia pancreatica exocrina": "Pancreatic Exocrine Insufficiency",
    "insuficiencia exocrina pancreatica": "Pancreatic Exocrine Insufficiency",
    "pancreatic exocrine insufficiency": "Pancreatic Exocrine Insufficiency",
    "desnutricion": "Malnutrition",
    "poor nutritional status": "Malnutrition",
    "metastasis": "Metastasis",
    "metastases": "Metastasis",
    "metastasis extra-pancreaticas": "Extrapancreatic Metastases",
    "metastasis viscerales": "Visceral Metastases",
    "biliary obstruction": "Biliary Obstruction",
    "obstruccion de un conducto biliar": "Biliary Obstruction",
    "obstruccion de conductos biliares": "Biliary Obstruction",
    "gastric outlet obstruction": "Gastric Outlet Obstruction",
    "gastric outlet/duodenal obstruction": "Gastric Outlet Obstruction",
    "obstruccion del estomago": "Gastric Outlet Obstruction",
    "duodenal obstruction": "Duodenal Obstruction",
    "vte": "Venous Thromboembolism",
    "venous thromboembolism": "Venous Thromboembolism",
    "enfermedad tromboembolica": "Venous Thromboembolism",
    "thromboembolic disease": "Venous Thromboembolism",
    "trombosis venosa profunda": "Deep Vein Thrombosis",
    "recidiva local": "Local Recurrence",
    "locally recurrent pancreatic cancer": "Local Recurrence",
    "local recurrence": "Local Recurrence",
    "recidiva cerca de la zona donde se extirpo el pancreas": "Local Recurrence",
    "hcv infection": "Hepatitis C Virus Infection",
    "hepatitis c virus": "Hepatitis C Virus Infection",
    "hepatitis c virus (hcv) infection": "Hepatitis C Virus Infection",
    "hepatitis b virus": "Hepatitis B Virus Infection",
    "hepatitis b infection": "Hepatitis B Virus Infection",
    "hepatitis b virus (hbv) infection": "Hepatitis B Virus Infection",
    "pancreatitis": "Pancreatitis",
    "acute pancreatitis": "Acute Pancreatitis",
    "chronic pancreatitis": "Chronic Pancreatitis",
    "benign chronic pancreatitis": "Chronic Pancreatitis",
    "autoimmune pancreatitis": "Autoimmune Pancreatitis",
    "hereditary pancreatitis": "Hereditary Pancreatitis",
    "peutz-jeghers syndrome": "Peutz-Jeghers Syndrome",
}

MOLECULAR_ALIASES = {
    "ca 19-9": "CA19-9",
    "ca19.9": "CA19-9",
    "ca19-9": "CA19-9",
    "ca 19-9 levels": "CA19-9",
    "high serum ca 19-9": "CA19-9",
    "ca 19-9 aumentando": "CA19-9",
    "cea": "CEA",
    "ace": "CEA",
    "ca-125": "CA-125",
    "reactive c protein": "C-Reactive Protein",
    "albumin levels": "Serum Albumin",
    "lactate dehydrogenase": "Lactate Dehydrogenase",
    "circulating tumor dna": "Circulating Tumor DNA",
    "circulating tumor cells": "Circulating Tumor Cells",
    "pd-l1": "PD-L1",
    "tmb-h": "TMB-H",
    "tmb-h [>=10 mut/mb": "TMB-H",
    "tmb-h [>=10 mut/mb]": "TMB-H",
    "tumor mutational burden (tmb)": "Tumor Mutational Burden (TMB)",
    "msi": "MSI",
    "microsatellite instability": "MSI",
    "microsatellite instability (msi)": "MSI",
    "msi-h": "MSI-H",
    "dmmr": "dMMR",
    "mismatch repair deficiency (dmmr)": "dMMR",
    "msi-h/dmmr": "MSI-H/dMMR",
    "mismatch repair deficiency (dmmr) and/or high microsatellite instability (msi-h)": "MSI-H/dMMR",
    "microsatellite instability high/mismatch repair deficient": "MSI-H/dMMR",
    "mir-3173-5p": "miR-3173-5p",
    "adn": "DNA",
    "dna": "DNA",
    "ribonucleotida reductasa": "Ribonucleotide Reductase",
    "ribonucleotide reductase": "Ribonucleotide Reductase",
    "insulina": "Insulin",
    "krasg12d": "KRAS G12D",
    "kras g12d": "KRAS G12D",
    "kras g12d mutation": "KRAS G12D",
    "kras g12c": "KRAS G12C",
    "kras g12c variant": "KRAS G12C",
    "kras g12v": "KRAS G12V",
    "kras g12r": "KRAS G12R",
    "kras q61h": "KRAS Q61H",
    "kras-wild type": "KRAS Wild-Type",
    "kras-wt": "KRAS Wild-Type",
    "kras": "KRAS",
    "kras mutation": "KRAS",
    "kras mutations": "KRAS",
    "mutacion en el gen atm": "ATM",
    "atm": "ATM",
    "mutaciones en los genes brca1 y brca2": "BRCA1/2",
    "brca1/2": "BRCA1/2",
    "brca 1/2 mutations": "BRCA1/2",
    "brca1/2 mutation": "BRCA1/2",
    "brca": "BRCA1/2",
    "brca mutations": "BRCA1/2",
    "brca1": "BRCA1",
    "brca2": "BRCA2",
    "mutacion en el gen palb2": "PALB2",
    "palb2": "PALB2",
    "palb2 mutation": "PALB2",
    "mutacion en el gen cdkn2a": "CDKN2A",
    "cdkn2a": "CDKN2A",
    "cdkn2a/b loss": "CDKN2A/B Loss",
    "mutacion en el gen stk11": "STK11",
    "stk11": "STK11",
    "mutacion en el gen tp53": "TP53",
    "tp53": "TP53",
    "tp53 mutational status": "TP53",
    "p53": "TP53",
    "loss of p53 function": "TP53 Loss of Function",
    "mutaciones en los genes mlh1, msh2, msh6 y pms2": "MMR Genes (MLH1/MSH2/MSH6/PMS2)",
    "mlh1": "MLH1",
    "msh2": "MSH2",
    "msh6": "MSH6",
    "pms2": "PMS2",
    "smad4": "SMAD4",
    "alk": "ALK",
    "ret": "RET",
    "ros1": "ROS1",
    "ros": "ROS1",
    "nrg1": "NRG1",
    "pik3ca": "PIK3CA",
    "bap1": "BAP1",
    "arid1b": "ARID1B",
    "dpyd": "DPYD",
    "hent1": "hENT1",
    "hent2": "hENT2",
    "hcnt1": "hCNT1",
    "hcnt2": "hCNT2",
    "hcnt3": "hCNT3",
    "cda": "CDA",
    "rrm1": "RRM1",
    "dck": "dCK",
    "acsl4": "ACSL4",
    "ces2": "CES2",
    "ngf": "NGF",
    "muc1": "MUC1",
    "cd44": "CD44",
    "sox2": "SOX2",
    "icam-1": "ICAM-1",
    "ccl2": "CCL2",
    "ccl5": "CCL5",
    "fasn": "FASN",
    "hif-1α": "HIF-1alpha",
    "ntrk": "NTRK Fusions",
    "ntrk1/2/3": "NTRK Fusions",
    "ntrk fusions": "NTRK Fusions",
    "ntrk rearrangements": "NTRK Fusions",
    "braf": "BRAF",
    "braf mutations": "BRAF",
    "braf v600e mutation": "BRAF V600E",
    "her2": "HER2",
    "her2/3": "HER2",
    "her2 overexpression": "HER2 Amplification/Overexpression",
    "her2 amplification": "HER2 Amplification/Overexpression",
    "amplifications (her2)": "HER2 Amplification/Overexpression",
    "idh1/2": "IDH1/2",
    "idh1 mutations": "IDH1 Mutation",
    "fgfr": "FGFR1-3",
    "fgfr1-3": "FGFR1-3",
    "fgfr2": "FGFR2",
    "fgfr2 fusions": "FGFR2 Fusions",
    "mutaciones de la linea germinal": "Germline Mutations",
    "germline mutations": "Germline Mutations",
    "mutacion de la linea germinal causante de cancer": "Germline Mutations",
    "mutaciones hereditarias": "Germline Mutations",
    "inherited pathogenic variants in cancer susceptibility genes": "Germline Mutations",
    "germline pathogenic variant in a cancer susceptibility gene": "Germline Mutations",
    "ddr/hrr pathways": "DDR/HRR Pathways",
    "ddr deficiency": "DDR/HRR Pathways",
    "notch2 signaling": "Notch2 Signaling",
    "swi/snf complex": "SWI/SNF Complex",
    "ppp": "Pentose Phosphate Pathway",
    "emt": "Epithelial-Mesenchymal Transition (EMT)",
    "emt phenotypes": "Epithelial-Mesenchymal Transition (EMT)",
}

STAGE_ALIASES = {
    "estadio 0": "Stage 0",
    "stage 0": "Stage 0",
    "estadio i": "Stage I",
    "stage i": "Stage I",
    "estadio ia": "Stage IA",
    "stage ia": "Stage IA",
    "estadio ib": "Stage IB",
    "stage ib": "Stage IB",
    "estadio ii": "Stage II",
    "estadio 2": "Stage II",
    "estadio ii (b2 de duke)": "Stage II",
    "stage ii": "Stage II",
    "estadio iia": "Stage IIA",
    "stage iia": "Stage IIA",
    "estadio iib": "Stage IIB",
    "stage iib": "Stage IIB",
    "estadio iii": "Stage III",
    "estadio 3": "Stage III",
    "estadio iii (c de duke)": "Stage III",
    "stage iii": "Stage III",
    "stage iii pancreatic cancer ajcc v8": "Stage III",
    "estadio iv": "Stage IV",
    "estadio 4": "Stage IV",
    "estadio iv de adenocarcinoma de pancreas": "Stage IV",
    "stage iv": "Stage IV",
    "stage iv pancreatic cancer ajcc v8": "Stage IV",
    "stage iii to iv disease": "Stage III-IV",
    "tumor primario t4": "T4 Stage",
    "ganglionar n0": "N0 Stage",
    "ganglionar n1": "N1 Stage",
    "ganglionar n2": "N2 Stage",
    "grado g1": "Grade G1",
    "grado g2": "Grade G2",
    "locally advanced disease": "Locally Advanced",
    "locally advanced": "Locally Advanced",
    "metastatic disease": "Metastatic",
    "m1 disease": "Metastatic",
    "locally advanced/metastatic disease": "Locally Advanced or Metastatic",
    "locally advanced or metastatic disease": "Locally Advanced or Metastatic",
    "borderline-resectable disease": "Borderline Resectable",
    "borderline resectable disease": "Borderline Resectable",
    "reseccion limitrofe": "Borderline Resectable",
    "resecable limitrofe": "Borderline Resectable",
    "resecabilidad limitrofe": "Borderline Resectable",
    "br": "Borderline Resectable",
    "br-pv": "Borderline Resectable",
    "reseccion irresecable": "Unresectable",
    "irresecable": "Unresectable",
    "resecabilidad": "Resectability",
    "ecog performance status (ps)": "ECOG Performance Status",
    "estado funcional": "ECOG Performance Status",
    "estado funcional 0 o 1": "ECOG PS 0-1",
    "estado funcional 0, 1 o 2": "ECOG PS 0-2",
    "estado funcional 3": "ECOG PS 3",
    "ecog 4": "ECOG PS 4",
    "estado funcional deficiente": "Poor Performance Status",
    "poor ps": "Poor Performance Status",
}

PROCEDURE_ALIASES = {
    "radioterapia": "Radiotherapy",
    "radiotherapy": "Radiotherapy",
    "radiation therapy": "Radiotherapy",
    "radiation": "Radiotherapy",
    "rt": "Radiotherapy",
    "radioterapia corporal estereotactica (sbrt)": "Stereotactic Body Radiotherapy (SBRT)",
    "radioterapia corporal estereotactica (rtce)": "Stereotactic Body Radiotherapy (SBRT)",
    "radioterapia corporal estereotactica": "Stereotactic Body Radiotherapy (SBRT)",
    "sbrt": "Stereotactic Body Radiotherapy (SBRT)",
    "radioterapia de haz externo (rthe)": "External Beam Radiotherapy",
    "radioterapia de haz externo": "External Beam Radiotherapy",
    "intensity-modulated rt (imrt)": "Intensity-Modulated Radiotherapy (IMRT)",
    "intensity-modulated rt": "Intensity-Modulated Radiotherapy (IMRT)",
    "imrt": "Intensity-Modulated Radiotherapy (IMRT)",
    "3d-crt": "3D Conformal Radiotherapy",
    "3-d conformal rt": "3D Conformal Radiotherapy",
    "mri": "MRI",
    "rm": "MRI",
    "irm": "MRI",
    "magnetic resonance imaging": "MRI",
    "hepatic mri": "Hepatic MRI",
    "liver mri": "Hepatic MRI",
    "ct": "Computed Tomography (CT)",
    "tc": "Computed Tomography (CT)",
    "computed tomography": "Computed Tomography (CT)",
    "computed tomography (ct)": "Computed Tomography (CT)",
    "tomografia computarizada": "Computed Tomography (CT)",
    "body ct scan": "Computed Tomography (CT)",
    "ct imaging": "Computed Tomography (CT)",
    "mdct": "Multidetector CT (MDCT)",
    "multidetector ct (mdct) angiography": "Multidetector CT (MDCT)",
    "exploraciones con protocolo pancreatico": "Pancreatic Protocol CT",
    "pdac-oriented ct scan": "Pancreatic Protocol CT",
    "pancreatic ct of abdomen": "Pancreatic Protocol CT",
    "tc con medio de contraste de torax y pelvis": "Contrast-Enhanced Chest-Abdomen-Pelvis CT",
    "multiphasic contrast-enhanced thoracic-abdominal and pelvic ct": "Contrast-Enhanced Chest-Abdomen-Pelvis CT",
    "tc de torax": "Chest CT",
    "chest ct": "Chest CT",
    "tc o rm": "CT/MRI",
    "ct/mri": "CT/MRI",
    "pet/ct": "PET/CT",
    "pet-ct": "PET/CT",
    "petct": "PET/CT",
    "tep/tc": "PET/CT",
    "pet/ct scan": "PET/CT",
    "positron emission tomography/computed tomography": "PET/CT",
    "fdg-pet": "PET",
    "pet scan": "PET",
    "tep": "PET",
    "positron emission tomography": "PET",
    "pet/mri": "PET/MRI",
    "tep/rm": "PET/MRI",
    "pet/mri scan": "PET/MRI",
    "eus": "Endoscopic Ultrasound (EUS)",
    "ecografia endoscopica": "Endoscopic Ultrasound (EUS)",
    "endoscopic ultrasound": "Endoscopic Ultrasound (EUS)",
    "eus-fna": "EUS-FNA",
    "fna": "EUS-FNA",
    "fine-needle aspiration": "EUS-FNA",
    "biopsia por puncion con aguja gruesa": "EUS-Guided Biopsy",
    "core biopsy": "EUS-Guided Biopsy",
    "eus-guided biopsy": "EUS-Guided Biopsy",
    "eus-guided needle biopsy": "EUS-Guided Biopsy",
    "biopsy by eus": "EUS-Guided Biopsy",
    "biopsy": "Biopsy",
    "biopsia": "Biopsy",
    "biopsias": "Biopsy",
    "ercp": "ERCP",
    "cpre": "ERCP",
    "endoscopic retrograde cholangiopancreatography": "ERCP",
    "mr-cp": "MRCP",
    "magnetic resonance cholangiopancreatography": "MRCP",
    "cirugia": "Pancreatic Surgery",
    "surgery": "Pancreatic Surgery",
    "cirugia pancreatica": "Pancreatic Surgery",
    "cirugia del cancer de pancreas": "Pancreatic Surgery",
    "reseccion": "Surgical Resection",
    "pancreatic resection": "Surgical Resection",
    "pancreatic resections": "Surgical Resection",
    "surgical resection": "Surgical Resection",
    "pancreatic cancer resection": "Surgical Resection",
    "pdac resection": "Surgical Resection",
    "srg": "Surgical Resection",
    "upfront surgery": "Upfront Surgery",
    "whipple procedure": "Pancreaticoduodenectomy (Whipple)",
    "pancreaticoduodenectomia": "Pancreaticoduodenectomy (Whipple)",
    "pancreatoduodenectomy": "Pancreaticoduodenectomy (Whipple)",
    "pancreaticoduodenectomy": "Pancreaticoduodenectomy (Whipple)",
    "tecnica de whipple": "Pancreaticoduodenectomy (Whipple)",
    "pancreatoduodenectomy (whipple technique)": "Pancreaticoduodenectomy (Whipple)",
    "pancreaticoduodenectomy (whipple procedure)": "Pancreaticoduodenectomy (Whipple)",
    "open or minimally invasive pancreatoduodenectomy (whipple procedure)": "Pancreaticoduodenectomy (Whipple)",
    "pancreatectomia distal": "Distal Pancreatectomy",
    "distal pancreatectomy": "Distal Pancreatectomy",
    "pancreatectomia distal ampliada": "Extended Distal Pancreatectomy",
    "pancreatectomia total": "Total Pancreatectomy",
    "total pancreatectomy": "Total Pancreatectomy",
    "esplenectomia en bloque": "En-Bloc Splenectomy",
    "en-bloc splenectomy": "En-Bloc Splenectomy",
    "cirugia laparoscopica": "Laparoscopic Surgery",
    "cirugia robotica": "Robotic Surgery",
    "laparoscopia": "Laparoscopy",
    "laparoscopy": "Laparoscopy",
    "pruebas geneticas": "Genetic Testing",
    "genetic testing": "Genetic Testing",
    "pruebas de funcion hepatica": "Liver Function Tests",
    "lfts": "Liver Function Tests",
    "pruebas de antigeno carbohidratado 19-9 (ca 19-9)": "CA19-9 Blood Test",
    "ca 19-9 levels determination": "CA19-9 Blood Test",
    "pruebas endoscopicas": "Endoscopic Testing",
    "cuidados paliativos": "Palliative Care",
    "palliative and supportive care": "Palliative Care",
    "cuidados de apoyo": "Supportive Care",
    "supportive care": "Supportive Care",
    "bsc": "Supportive Care",
    "symptom-directed care": "Supportive Care",
    "active symptom control": "Supportive Care",
    "apoyo nutricional": "Nutritional Support",
    "equipo multidisciplinario": "Multidisciplinary Team (MDT)",
    "mdt": "Multidisciplinary Team (MDT)",
    "multidisciplinary review": "Multidisciplinary Team (MDT)",
    "multidisciplinary consultation": "Multidisciplinary Team (MDT)",
    "multidisciplinary meetings/discussions": "Multidisciplinary Team (MDT)",
    "fisioterapia intensiva": "Intensive Physiotherapy",
    "seguimiento nutricional mediante aplicacion movil noom": "Nutritional Monitoring (Noom App)",
    "inmunoterapia con inhibidores de puntos de control": "Immune Checkpoint Inhibitors",
    "immune checkpoint blockade (icb)": "Immune Checkpoint Inhibitors",
    "immune checkpoint inhibitors": "Immune Checkpoint Inhibitors",
    "vacunacion con vacunas vivas o vivas atenuadas": "Live Attenuated Vaccination",
    "simultaneidad con inhibidores o inductores potentes del citocromo p450 3a4": "Co-administration with Strong CYP3A4 Modulators",
    "celiac plexus neurolysis": "Celiac Plexus Neurolysis",
    "celiac neurolysis": "Celiac Plexus Neurolysis",
    "reconstruccion": "Surgical Reconstruction",
}

ADVERSE_EVENT_AND_SYMPTOM_ALIASES = {
    "diarrea": "Diarrhea",
    "diarrhea": "Diarrhea",
    "diarrea grave": "Severe Diarrhea",
    "severe diarrhea": "Severe Diarrhea",
    "diarrea tardia": "Delayed Diarrhea",
    "diarrea temprana": "Early Diarrhea",
    "heces liquidas": "Watery Stools",
    "nauseas": "Nausea",
    "nausea": "Nausea",
    "nauseas graves": "Severe Nausea",
    "nauseas y vomitos": "Nausea and Vomiting",
    "nausea/vomiting": "Nausea and Vomiting",
    "vomitos": "Vomiting",
    "vomiting": "Vomiting",
    "vomitos graves": "Severe Vomiting",
    "fatiga": "Fatigue",
    "fatigue": "Fatigue",
    "cansancio": "Fatigue",
    "astenia": "Asthenia",
    "astenia/fatiga": "Asthenia",
    "astenia grave": "Severe Asthenia",
    "anemia": "Anemia",
    "anemia hemolitica": "Hemolytic Anemia",
    "neutropenia": "Neutropenia",
    "neutropenia grave": "Severe Neutropenia",
    "neutropenia febril": "Febrile Neutropenia",
    "trombocitopenia": "Thrombocytopenia",
    "trombocitopenia inmunoalergica": "Immunoallergic Thrombocytopenia",
    "trombocitopenia con anticuerpos antiplaquetarios": "Antibody-Mediated Thrombocytopenia",
    "purpura trombocitopenica": "Thrombocytopenic Purpura",
    "trombocitosis": "Thrombocytosis",
    "leucopenia": "Leukopenia",
    "linfopenia": "Lymphopenia",
    "linfocitopenia": "Lymphopenia",
    "pancitopenia": "Pancytopenia",
    "pancitopenia autoinmune": "Autoimmune Pancytopenia",
    "agranulocitosis": "Agranulocytosis",
    "mielosupresion": "Myelosuppression",
    "mielosupresion antes del comienzo del primer ciclo": "Pre-Treatment Myelosuppression",
    "baja cantidad de celulas sanguineas": "Cytopenias",
    "alopecia": "Alopecia",
    "alopecia (reversible)": "Alopecia",
    "anorexia": "Anorexia",
    "disminucion del apetito": "Anorexia",
    "fiebre": "Fever",
    "pirexia": "Fever",
    "pirexia sin infeccion": "Fever",
    "hiperpirexia": "Fever",
    "escalofrios": "Chills",
    "rigor": "Chills",
    "dolor": "Pain",
    "pain": "Pain",
    "dolor generalizado": "Pain",
    "dolor abdominal": "Abdominal Pain",
    "abdominal pain": "Abdominal Pain",
    "dolor abdominal superior": "Upper Abdominal Pain",
    "dolor de espalda": "Back Pain",
    "dolor de cabeza": "Headache",
    "cefalea": "Headache",
    "dolor de pecho": "Chest Pain",
    "dolor toracico": "Chest Pain",
    "sensacion de dolor toracico": "Chest Pain",
    "dolor de la pared toracica": "Chest Wall Pain",
    "dolor musculoesqueletico": "Musculoskeletal Pain",
    "dolor oseo": "Bone Pain",
    "dolor en las extremidades": "Extremity Pain",
    "dolor en los miembros": "Extremity Pain",
    "malestar en los miembros": "Extremity Discomfort",
    "dolor en la piel": "Skin Pain",
    "dolor mamario": "Breast Pain",
    "dolor facial": "Facial Pain",
    "dolor ocular": "Ocular Pain",
    "dolor de cuello": "Neck Pain",
    "dolor inguinal": "Inguinal Pain",
    "dolor costal": "Rib Pain",
    "dolor por metastasis": "Metastatic Pain",
    "estomatitis": "Mucositis / Stomatitis",
    "mucositis": "Mucositis / Stomatitis",
    "estomatitis / mucositis": "Mucositis / Stomatitis",
    "inflamacion de las mucosas": "Mucositis / Stomatitis",
    "ulceras bucales": "Oral Ulcers",
    "mucositis severa": "Severe Mucositis",
    "estrenimiento": "Constipation",
    "constipation": "Constipation",
    "disnea": "Dyspnea",
    "infecciones": "Infection",
    "infeccion": "Infection",
    "infection": "Infection",
    "aumento de las infecciones": "Infection",
    "infeccion sin neutropenia": "Non-Neutropenic Infection",
    "infecciones asociadas con neutropenia grave": "Neutropenic Infection",
    "infeccion del tracto urinario": "Urinary Tract Infection",
    "infeccion de las vias respiratorias altas": "Upper Respiratory Tract Infection",
    "infecciones por hongos": "Fungal Infection",
    "infeccion fungica": "Fungal Infection",
    "infecciones viricas": "Viral Infection",
    "infeccion viral": "Viral Infection",
    "infeccion relacionada con el cateter": "Catheter-Related Infection",
    "infeccion del lugar de inyeccion": "Injection Site Infection",
    "sepsis": "Sepsis",
    "sepsis neutropenica": "Neutropenic Sepsis",
    "sepsis (incluyendo sepsis y sepsis neutropenica)": "Sepsis",
    "hipotension": "Hypotension",
    "hipotension ortostatica": "Orthostatic Hypotension",
    "hipertension": "Hypertension",
    "neuropatia periferica": "Peripheral Neuropathy",
    "peripheral neuropathy": "Peripheral Neuropathy",
    "neuropathy": "Peripheral Neuropathy",
    "neuropatia sensorial": "Sensory Neuropathy",
    "neuropatia sensorial periferica con deterioro funcional antes del primer ciclo": "Pre-Treatment Sensory Neuropathy",
    "neuropatia motora": "Motor Neuropathy",
    "neuropatia periferica de grado 3": "Grade 3 Peripheral Neuropathy",
    "neuropatia periferica de grado 4": "Grade 4 Peripheral Neuropathy",
    "parestesia": "Paresthesia",
    "parestesias": "Paresthesia",
    "parestesia localizada persistente": "Persistent Paresthesia",
    "disestesia": "Dysesthesia",
    "hipoestesia": "Hypoesthesia",
    "hipoestesia oral": "Oral Hypoesthesia",
    "sindrome agudo de disestesia faringolaringea": "Acute Pharyngolaryngeal Dysesthesia",
    "ictericia": "Jaundice",
    "jaundice": "Jaundice",
    "obstructive jaundice": "Obstructive Jaundice",
    "esteatorrea": "Steatorrhea",
    "steatorrhoea": "Steatorrhea",
    "floating stools": "Floating Stools",
    "perdida de peso": "Weight Loss",
    "disminucion de peso": "Weight Loss",
    "weight loss": "Weight Loss",
    "continuous weight loss": "Weight Loss",
    "tabaquismo": "Smoking",
    "smoking": "Smoking",
    "cigarette smoking": "Smoking",
    "tobacco": "Smoking",
    "obesity": "Obesity",
    "overweight": "Obesity",
    "antecedentes familiares de cancer de pancreas": "Family History of Pancreatic Cancer",
    "antecedentes familiares de cancer": "Family History of Pancreatic Cancer",
    "family history of pancreatic ductal adenocarcinoma": "Family History of Pancreatic Cancer",
    "aumento de las enzimas hepaticas": "Elevated Liver Enzymes",
    "aumento de transaminasas": "Elevated Liver Transaminases",
    "aumento de las transaminasas": "Elevated Liver Transaminases",
    "aumento de las transaminasas hepaticas": "Elevated Liver Transaminases",
    "aumento de las transaminasas hepaticas (ast y alt)": "Elevated Liver Transaminases",
    "aumento de las transaminasas (alt y ast)": "Elevated Liver Transaminases",
    "aumento de la alanina aminotransferasa": "Elevated ALT",
    "elevated levels of alanine aminotransferase": "Elevated ALT",
    "aumento de la aspartato aminotransferasa": "Elevated AST",
    "aumento de la bilirrubina": "Hyperbilirubinemia",
    "aumento de bilirrubina": "Hyperbilirubinemia",
    "aumento de la bilirrubina en sangre": "Hyperbilirubinemia",
    "hiperbilirrubinemia": "Hyperbilirubinemia",
    "aumento de la fosfatasa alcalina": "Elevated Alkaline Phosphatase",
    "aumento de fosfatasa alcalina": "Elevated Alkaline Phosphatase",
    "aumento de la fosfatasa alcalina en sangre": "Elevated Alkaline Phosphatase",
    "fosfatasa alcalina": "Elevated Alkaline Phosphatase",
    "aumento de la gamma-glutamil-transferasa (ggt)": "Elevated GGT",
    "aumento de gamma-glutamil-transferasa (ggt)": "Elevated GGT",
    "aumento de la gamma glutamil transferasa": "Elevated GGT",
    "aumento de la gamma-glutamiltransferasa": "Elevated GGT",
    "aumento de la creatinina en sangre": "Elevated Blood Creatinine",
    "aumento de creatinina": "Elevated Blood Creatinine",
    "aumento de la amilasa": "Elevated Amylase",
    "aumento de la lipasa": "Elevated Lipase",
    "aumento de la lactato deshidrogenasa en sangre": "Elevated LDH",
    "deshidratacion": "Dehydration",
    "edema": "Edema",
    "edema periferico": "Peripheral Edema",
    "retencion de liquidos": "Fluid Retention",
    "ansiedad": "Anxiety",
    "anxiety": "Anxiety",
    "depresion": "Depression",
    "depression": "Depression",
    "insomnio": "Insomnia",
    "mareos": "Dizziness",
    "somnolencia": "Somnolence",
    "confusion": "Confusion",
    "sincope": "Syncope",
    "nerviosismo": "Nervousness",
    "infarto de miocardio": "Myocardial Infarction / Ischemia",
    "isquemia miocardica/infarto": "Myocardial Infarction / Ischemia",
    "isquemia de miocardio": "Myocardial Infarction / Ischemia",
    "arritmia": "Arrhythmia",
    "arritmias ventriculares": "Ventricular Arrhythmia",
    "torsade de pointes": "Torsade de Pointes",
    "bradicardia": "Bradycardia",
    "taquicardia": "Tachycardia",
    "taquicardia supraventricular": "Supraventricular Tachycardia",
    "insuficiencia cardiaca": "Heart Failure",
    "insuficiencia cardiaca congestiva": "Congestive Heart Failure",
    "disfuncion ventricular izquierda": "Left Ventricular Dysfunction",
    "bloqueo auriculoventricular": "Atrioventricular Block",
    "bloqueo auriculoventricular 1": "Atrioventricular Block",
    "extravasion": "Injection Site Extravasation",
    "manifestaciones en la piel": "Skin Manifestations",
    "trastornos del habla generalmente transitorios": "Transient Speech Disorder",
    "trastorno visual": "Visual Impairment",
    "reduccion pasajera de la agudeza visual": "Decreased Visual Acuity",
    "reduccion de la agudeza visual": "Decreased Visual Acuity",
    "trastornos del campo visual": "Visual Field Disorder",
    "perdida pasajera de la vision": "Transient Vision Loss",
    "deterioro sensorial": "Sensory Impairment",
    "parestesia que puede interferir en las actividades funcionales": "Functional-Impairing Paresthesia",
    "tirantez/opresion/molestia/dolor en la garganta o en el pecho": "Throat or Chest Tightness/Pain",
    "alteracion de la funcion de los nervios craneales": "Cranial Nerve Dysfunction",
    "sensacion anomala en la lengua": "Abnormal Tongue Sensation",
    "perdida del reflejo del tendon profundo": "Loss of Deep Tendon Reflex",
    "alteracion de la sintesis del adn": "Impaired DNA Synthesis",
    "parada cardiaca": "Cardiac Arrest",
    "angina de pecho": "Angina Pectoris",
    "arterioespasmo coronario": "Coronary Arteriospasm",
    "prolongacion del intervalo qt": "QT Prolongation",
    "mialgia": "Myalgia",
    "artralgia": "Arthralgia",
    "arthralgia": "Arthralgia",
    "calambres": "Muscle Cramps",
    "calambres musculares": "Muscle Cramps",
    "debilidad muscular": "Muscle Weakness",
    "espasmos musculares": "Muscle Spasms",
    "espasmos mandibulares": "Jaw Spasms",
    "contracciones musculares involuntarias": "Involuntary Muscle Contractions",
    "alteraciones del gusto": "Dysgeusia",
    "disgeusia": "Dysgeusia",
    "sequedad de boca": "Dry Mouth",
    "disfagia": "Dysphagia",
    "dispepsia": "Dyspepsia",
    "indigestion": "Dyspepsia",
    "distension abdominal": "Abdominal Distension",
    "reflujo gastroesofagico": "Gastroesophageal Reflux",
    "rectorragia": "Rectal Bleeding",
    "reaccion alergica": "Hypersensitivity / Allergic Reaction",
    "reacciones alergicas": "Hypersensitivity / Allergic Reaction",
    "alergia": "Hypersensitivity / Allergic Reaction",
    "alergias o reacciones alergicas": "Hypersensitivity / Allergic Reaction",
    "hipersensibilidad": "Hypersensitivity / Allergic Reaction",
    "hipersensibilidad retardada": "Delayed Hypersensitivity",
    "hipersensibilidad al oxaliplatino": "Oxaliplatin Hypersensitivity",
    "reaccion anafilactica": "Anaphylaxis",
    "reacciones anafilacticas o anafilactoides": "Anaphylaxis",
    "choque anafilactico": "Anaphylaxis",
    "broncoespasmo": "Bronchospasm",
    "hemorragia": "Hemorrhage",
    "hemorragias": "Hemorrhage",
    "bleeding": "Hemorrhage",
    "hemorrhage": "Hemorrhage",
    "hemorragias nasales": "Epistaxis",
    "neumonitis": "Interstitial Lung Disease / Pneumonitis",
    "pneumonitis": "Interstitial Lung Disease / Pneumonitis",
    "neumonitis intersticial": "Interstitial Lung Disease / Pneumonitis",
    "enfermedad pulmonar intersticial": "Interstitial Lung Disease / Pneumonitis",
    "interstitial lung disease (ild)": "Interstitial Lung Disease / Pneumonitis",
    "neumonitis inducida por la radiacion": "Radiation Pneumonitis",
    "lesion por radiacion": "Radiation Injury",
    "esofagitis": "Esophagitis",
    "colitis": "Colitis",
    "colitis isquemica": "Ischemic Colitis",
    "colitis pseudomembranosa": "Pseudomembranous Colitis",
    "colitis por citomegalovirus": "Cytomegalovirus Colitis",
    "obstruccion intestinal": "Intestinal Obstruction",
    "ileo paralitico": "Paralytic Ileus",
    "erupcion cutanea": "Rash",
    "erupcion": "Rash",
    "rash": "Rash",
    "erupcion cutanea alergica": "Allergic Rash",
    "erupcion cutanea de tipo alergico": "Allergic Rash",
    "reaccion en la piel": "Skin Reaction",
    "reacciones cutaneas graves": "Severe Skin Reactions",
    "prurito": "Pruritus",
    "picor": "Pruritus",
    "prurito generalizado": "Pruritus",
    "piel seca": "Dry Skin",
    "urticaria": "Urticaria",
    "dermatitis": "Dermatitis",
    "dermatitis acneiforme": "Acneiform Dermatitis",
    "sindrome mano-pie": "Hand-Foot Syndrome",
    "sindrome de eritrodisestesia palmoplantar": "Hand-Foot Syndrome",
    "sindrome de steven-johnson": "Stevens-Johnson Syndrome",
    "necrolisis epidermica toxica": "Toxic Epidermal Necrolysis",
    "descamacion": "Desquamation",
    "ampollas en la piel": "Skin Blistering",
    "ulceracion": "Ulceration",
    "formacion de vesiculas y ampollas": "Vesiculobullous Rash",
    "pseudocelulitis": "Pseudocellulitis",
    "pustulosis exantematica generalizada aguda": "Acute Generalized Exanthematous Pustulosis",
    "reaccion a farmaco con eosinofilia y sintomas sistemicos (dress)": "DRESS Syndrome",
    "reaccion de fotosensibilidad": "Photosensitivity Reaction",
    "fenomeno de fotosensibilidad inducida por la radiacion": "Radiation Recall Photosensitivity",
    "hiperpigmentacion de la piel": "Skin Hyperpigmentation",
    "onicolisis": "Onycholysis",
    "onicomadesis": "Onychomadesis",
    "trastornos de las unas": "Nail Disorders",
    "cambio en las unas": "Nail Disorders",
    "escleroderma": "Scleroderma",
    "hipopotasemia": "Hypokalemia",
    "bajos niveles de potasio en sangre": "Hypokalemia",
    "bajos niveles de potasio y magnesio en sangre": "Hypokalemia and Hypomagnesemia",
    "hiponatremia": "Hyponatremia",
    "hyponatraemia": "Hyponatremia",
    "altos niveles de sodio en sangre": "Hypernatremia",
    "hipofosfatemia": "Hypophosphatemia",
    "hypophosphataemia": "Hypophosphatemia",
    "hipomagnesemia": "Hypomagnesemia",
    "hipocalcemia": "Hypocalcemia",
    "hiperglucemia": "Hyperglycemia",
    "altos niveles de glucosa en sangre": "Hyperglycemia",
    "aumento de la glucosa en sangre": "Hyperglycemia",
    "hipoglucemia": "Hypoglycemia",
    "hiperuricemia": "Hyperuricemia",
    "hipoalbuminemia": "Hypoalbuminemia",
    "hipovolemia": "Hypovolemia",
    "acidosis metabolica": "Metabolic Acidosis",
    "polidipsia": "Polydipsia",
    "insuficiencia renal": "Renal Failure",
    "insuficiencia renal aguda": "Acute Kidney Injury",
    "proteinuria": "Proteinuria",
    "proteinas en la orina": "Proteinuria",
    "hematuria": "Hematuria",
    "sangre en la orina": "Hematuria",
    "sindrome uremico hemolitico": "Hemolytic Uremic Syndrome",
    "sindrome de lisis tumoral": "Tumor Lysis Syndrome",
    "hepatoxicidad grave": "Severe Hepatotoxicity",
    "fallo hepatico": "Hepatic Failure",
    "esteatosis hepatica": "Hepatic Steatosis",
    "esteatohepatitis": "Steatohepatitis",
    "sindrome de obstruccion sinusoidal hepatica": "Hepatic Sinusoidal Obstruction Syndrome",
    "hipertension portal": "Portal Hypertension",
    "hiperplasia nodular regenerativa": "Nodular Regenerative Hyperplasia",
    "fibrosis perisinusoidal": "Perisinusoidal Fibrosis",
    "peliosis hepatica": "Peliosis Hepatis",
    "hepatomegalia": "Hepatomegaly",
    "colangitis": "Cholangitis",
    "sudoracion": "Hyperhidrosis",
    "hiperhidrosis": "Hyperhidrosis",
    "sudoracion nocturna": "Night Sweats",
    "rubor": "Flushing",
    "enrojecimiento": "Flushing",
    "sindrome pseudogripal": "Flu-Like Symptoms",
    "enfermad seudogripal": "Flu-Like Symptoms",
    "sintomas parecidos a los de la gripe": "Flu-Like Symptoms",
    "gripe": "Influenza",
    "herpes zoster": "Herpes Zoster",
    "herpes simple": "Herpes Simplex",
    "candidiasis": "Candidiasis",
    "candidiasis oral": "Oral Candidiasis",
    "candidiasis gastrointestinal": "Gastrointestinal Candidiasis",
    "neumonia": "Pneumonia",
    "neumonia por pneumocystis jirovecii": "Pneumocystis Jirovecii Pneumonia",
    "aspergilosis broncopulmonar": "Bronchopulmonary Aspergillosis",
    "candida sistemica": "Systemic Candidiasis",
    "reactivacion de la hepatitis b": "Hepatitis B Reactivation",
    "foliculitis": "Folliculitis",
    "sinusitis": "Sinusitis",
    "nasofaringitis": "Nasopharyngitis",
    "celulitis": "Cellulitis",
    "sindrome colinergico": "Cholinergic Syndrome",
    "sindrome colinergico agudo": "Cholinergic Syndrome",
    "sindrome colinergico agudo transitorio grave": "Cholinergic Syndrome",
    "miosis": "Miosis",
    "aumento de la salivacion": "Hypersalivation",
    "extravasacion": "Injection Site Extravasation",
    "extravasacion en el lugar de inyeccion": "Injection Site Extravasation",
    "reacciones en el lugar de la inyeccion": "Injection Site Reaction",
    "reaccion en el lugar de inyeccion": "Injection Site Reaction",
    "reacciones en el lugar de la perfusion": "Infusion Site Reaction",
    "inflamacion en el lugar de inyeccion": "Injection Site Inflammation",
    "erupcion en el lugar de inyeccion": "Injection Site Rash",
    "lactancia": "Lactation / Breastfeeding",
    "pacientes en periodo de lactancia": "Lactation / Breastfeeding",
    "gemcitabine resistance": "Gemcitabine Resistance",
    "gem resistance": "Gemcitabine Resistance",
    "resistance to mrtx1133": "MRTX1133 Resistance",
    "acquired resistance to mrtx1133": "MRTX1133 Resistance",
    "trombosis": "Thrombosis",
    "thrombosis": "Thrombosis",
    "eventos tromboticos": "Thrombosis",
    "trombosis/embolia": "Thromboembolism",
    "embolos pulmonares": "Pulmonary Embolism",
    "derrame pleural": "Pleural Effusion",
    "edema pulmonar": "Pulmonary Edema",
    "sindrome de distres respiratorio del adulto": "Acute Respiratory Distress Syndrome (ARDS)",
    "eosinofilia pulmonar": "Pulmonary Eosinophilia",
    "tos": "Cough",
    "rinitis": "Rhinitis",
    "conjuntivitis": "Conjunctivitis",
    "neuritis optica": "Optic Neuritis",
    "ototoxicidad": "Ototoxicity",
    "tinnitus": "Tinnitus",
    "vertigo": "Vertigo",
    "otalgia": "Otalgia",
    "sindrome de fuga capilar": "Capillary Leak Syndrome",
    "microangiopatia trombotica": "Thrombotic Microangiopathy",
    "accidente cerebrovascular": "Stroke",
    "sindrome de encefalopatia posterior reversible": "Posterior Reversible Encephalopathy Syndrome (PRES)",
    "vasculitis periferica": "Peripheral Vasculitis",
    "gangrena": "Gangrene",
    "inmunosupresion": "Immunosuppression",
    "bloqueo neuromuscular": "Neuromuscular Blockade",
    "coagulacion diseminada intravascular (cdi)": "Disseminated Intravascular Coagulation (DIC)",
    "leucemia secundaria": "Secondary Leukemia",
    "necrosis tumoral": "Tumor Necrosis",
    "ataxia": "Ataxia",
    "afasia": "Aphasia",
    "disartria": "Dysarthria",
    "temblores": "Tremor",
    "neuralgia del trigemino": "Trigeminal Neuralgia",
    "signo de lhermitte": "Lhermitte Sign",
    "paralisis de las cuerdas vocales": "Vocal Cord Paralysis",
    "alteracion de la marcha": "Gait Disturbance",
    "trastorno de la marcha": "Gait Disturbance",
    "trastornos del equilibro": "Balance Disorder",
    "alteracion de la coordinacion": "Coordination Disorder",
    "edema macular cistoide": "Cystoid Macular Edema",
    "paralisis multiple de los pares craneales": "Multiple Cranial Nerve Palsy",
    "disminucion del estado funcional": "Decreased Performance Status",
    "descenso del hematocrito": "Decreased Hematocrit",
    "descenso del recuento de globulos rojos": "Decreased Red Blood Cell Count",
    "aumento de la temperatura corporal": "Increased Body Temperature",
    "contusion": "Contusion",
    "toxicidad asociada a la radioterapia": "Radiation-Associated Toxicity",
    "toxicidad cutanea tardia": "Late Skin Toxicity",
    "efectos cardiacos": "Cardiac Toxicity",
    "insuficiencia cardiocirculatoria": "Cardiocirculatory Failure",
    "sensacion periférica de frio": "Peripheral Coldness",
    "sensacion periferica de frio": "Peripheral Coldness",
}

TRIAL_ALIASES = {
    "polo trial": "POLO Trial",
    "lap-07 trial": "LAP-07 Trial",
    "codebreak 100 trial": "CodeBreaK 100",
    "codebreak 100 (strickler 2023)": "CodeBreaK 100",
    "napoli-3": "NAPOLI-3",
    "conko-003": "CONKO-003",
    "conko-007": "CONKO-007",
    "norpact": "NORPACT",
    "cassandra": "CASSANDRA",
    "csgo-hbp-015": "CSGO-HBP-015",
    "mpaca-3 (go 2021)": "MPACA-3",
    "gempax (de la fouchardiere 2024)": "GEMPAX",
    "eclipse (le 2019)": "ECLIPSE",
    "sequoia (hecht 2021)": "SEQUOIA",
}

ENTITY_ALIASES = {}
for _bucket in (
    DRUG_ALIASES,
    REGIMEN_ALIASES,
    DISEASE_ALIASES,
    MOLECULAR_ALIASES,
    STAGE_ALIASES,
    PROCEDURE_ALIASES,
    ADVERSE_EVENT_AND_SYMPTOM_ALIASES,
    TRIAL_ALIASES,
):
    ENTITY_ALIASES.update(_bucket)

print(f"Total reglas de resolución de alias cargadas: {len(ENTITY_ALIASES)}")

**4. Overrides semánticos de tipo y filtro de entidades genéricas/ruido**
Reasigna entidades hacia tipos más específicos (`GENE`, `BIOMARKER`, `PATHWAY`, `SYMPTOM`, `RISK_FACTOR`) y lista artefactos vagos para cuarentena.

In [ ]:
# Reasignación semántica determinista hacia los 13 tipos del grafo (resuelve conflictos y promueve GENES/BIOMARKERS/PATHWAYS)
ENTITY_TYPE_OVERRIDES = {
    # Biomarkers
    "CA19-9": "BIOMARKER",
    "CEA": "BIOMARKER",
    "CA-125": "BIOMARKER",
    "PD-L1": "BIOMARKER",
    "TMB-H": "BIOMARKER",
    "Tumor Mutational Burden (TMB)": "BIOMARKER",
    "MSI": "BIOMARKER",
    "MSI-H": "BIOMARKER",
    "dMMR": "BIOMARKER",
    "MSI-H/dMMR": "BIOMARKER",
    "C-Reactive Protein": "BIOMARKER",
    "Serum Albumin": "BIOMARKER",
    "Lactate Dehydrogenase": "BIOMARKER",
    "Circulating Tumor DNA": "BIOMARKER",
    "Circulating Tumor Cells": "BIOMARKER",
    "miR-3173-5p": "BIOMARKER",
    "Fasting Blood Glucose": "BIOMARKER",
    "Vitamin D": "BIOMARKER",
    "PET SUV": "BIOMARKER",
    "Tumor DNA Content": "BIOMARKER",
    "High-Density Lipoprotein Cholesterol": "BIOMARKER",
    "TIGAR Expression Levels": "BIOMARKER",
    # Genes
    "KRAS": "GENE",
    "BRCA1": "GENE",
    "BRCA2": "GENE",
    "BRCA1/2": "GENE",
    "PALB2": "GENE",
    "TP53": "GENE",
    "ATM": "GENE",
    "CDKN2A": "GENE",
    "SMAD4": "GENE",
    "MLH1": "GENE",
    "MSH2": "GENE",
    "MSH6": "GENE",
    "PMS2": "GENE",
    "MMR Genes (MLH1/MSH2/MSH6/PMS2)": "GENE",
    "STK11": "GENE",
    "ALK": "GENE",
    "RET": "GENE",
    "ROS1": "GENE",
    "NRG1": "GENE",
    "PIK3CA": "GENE",
    "BAP1": "GENE",
    "ARID1B": "GENE",
    "DPYD": "GENE",
    "hENT1": "GENE",
    "hENT2": "GENE",
    "hCNT1": "GENE",
    "hCNT2": "GENE",
    "hCNT3": "GENE",
    "CDA": "GENE",
    "RRM1": "GENE",
    "dCK": "GENE",
    "ACSL4": "GENE",
    "CES2": "GENE",
    "NGF": "GENE",
    "MUC1": "GENE",
    "CD44": "GENE",
    "SOX2": "GENE",
    "ICAM-1": "GENE",
    "CCL2": "GENE",
    "CCL5": "GENE",
    "FASN": "GENE",
    "BRAF": "GENE",
    "HER2": "GENE",
    "IDH1/2": "GENE",
    "FGFR1-3": "GENE",
    "FGFR2": "GENE",
    "PRSS1": "GENE",
    "SPINK1": "GENE",
    "CTRC": "GENE",
    "CFTR": "GENE",
    "CASR": "GENE",
    "SOS1": "GENE",
    # Pathways
    "DDR/HRR Pathways": "PATHWAY",
    "Notch2 Signaling": "PATHWAY",
    "SWI/SNF Complex": "PATHWAY",
    "Pentose Phosphate Pathway": "PATHWAY",
    "Epithelial-Mesenchymal Transition (EMT)": "PATHWAY",
    # Risk Factors
    "Smoking": "RISK_FACTOR",
    "Obesity": "RISK_FACTOR",
    "Family History of Pancreatic Cancer": "RISK_FACTOR",
    "Hereditary Syndromes": "RISK_FACTOR",
    "Familial Syndromes": "RISK_FACTOR",
    "Gallstones": "RISK_FACTOR",
    # Symptoms
    "Jaundice": "SYMPTOM",
    "Obstructive Jaundice": "SYMPTOM",
    "Steatorrhea": "SYMPTOM",
    "Floating Stools": "SYMPTOM",
    "Local Obstructive Symptoms": "SYMPTOM",
    "Abdominal Symptoms": "SYMPTOM",
    # Cross-type conflict resolution
    "Chemoradiation": "PROCEDURE",
    "Chemoradiation / SBRT": "PROCEDURE",
    "Diabetes Mellitus": "DISEASE",
    "Type 2 Diabetes Mellitus": "DISEASE",
    "New-Onset Diabetes Mellitus": "DISEASE",
    "Pancreatitis": "DISEASE",
    "Biliary Obstruction": "DISEASE",
    "Gastric Outlet Obstruction": "DISEASE",
    "Duodenal Obstruction": "DISEASE",
    "Malnutrition": "DISEASE",
    "Liver Metastases": "DISEASE",
    "Metastasis": "DISEASE",
    "Metastatic": "STAGE",
    "Locally Advanced": "STAGE",
    "Locally Advanced or Metastatic": "STAGE",
    "Borderline Resectable": "STAGE",
    "Unresectable": "STAGE",
    "Poor Performance Status": "STAGE",
    "Metastatic Pancreatic Cancer": "DISEASE",
    "Metastatic Pancreatic Ductal Adenocarcinoma": "DISEASE",
    "Locally Advanced Pancreatic Cancer": "DISEASE",
    "Locally Advanced Pancreatic Ductal Adenocarcinoma": "DISEASE",
    "Resectable Pancreatic Cancer": "DISEASE",
    "Adjuvant Chemotherapy": "REGIMEN",
    "Neoadjuvant Chemotherapy": "REGIMEN",
    "5-Fluorouracil/Leucovorin": "REGIMEN",
    "Live Attenuated Vaccines": "DRUG",
    "Yellow Fever Vaccine": "DRUG",
    "Inactivated Vaccines": "DRUG",
}

# Entidades genéricas, encabezados MedDRA o ruido de extracción que deben enviarse a cuarentena
GENERIC_NOISE_ENTITIES = {
    "pancreatic",
    "ampullary",
    "other",
    "otros",
    "otros farmacos anticancerosos",
    "investigational agents",
    "different neoadjuvant therapy",
    "systemic anticancer therapy",
    "other cytotoxic chemotherapies",
    "other fluoropyrimidine-based therapies",
    "cancer or advanced disease",
    "cancer de otros organos",
    "other extrapancreatic disease",
    "other - thrombus within vein (tumor, bland)",
    "other - venous collaterals",
    "recuento basal de neutrofilos < 1.500 celulas/mm3",
    "creatinine clearance < 45 ml/min",
    "creatinine clearance < 60 ml/min",
    "albumin <35 g/l",
    "treatment-related adverse events",
    "grade 3 events",
    "adverse effects",
    "efectos adversos grado 3-4",
    "efectos secundarios clasicos de la quimioterapia",
    "problemas de salud a largo plazo",
    "problemas cutaneos",
    "estado de deterioro definitivo",
    "deterioro definitivo",
    "sintomas persistentes",
    "dificultades para llevar a cabo movimientos delicados",
    "infecciones e infestaciones",
    "trastornos del sistema nervioso",
    "trastornos de la piel y del tejido subcutaneo",
    "trastornos musculoesqueleticos y del tejido conjuntivo",
    "trastornos del oido y del laberinto",
    "trastornos cardiacos",
    "trastornos cardiovasculares",
    "trastornos de la piel",
    "trastornos de la pigmentacion",
    "suffering",
    "resistance",
    "drug resistance",
    "resistance to therapy",
    "some autoimmune diseases",
    "genetic abnormalities",
    "genetic and molecular alterations",
    "mutaciones en ciertos genes",
    "biological and conditional features",
    "anatomical nccn criteria",
    "high-quality dedicated imaging",
    "latest generation eus needle",
    "false-positive rate",
    "false-negative rate",
    "false negatives",
    "longer operative times",
    "suboptimal clearance",
    "rct measuring effectiveness of ethanol celiac plexus neurolysis",
    "tratamiento primario",
    "tratamiento complementario",
    "estudios de diagnostico por imagenes",
    "estudios de diagnostico por imagenes adicionales",
    "single-agent chemotherapy",
    "targeted therapy",
    "cytostatics",
    "immunotherapy",
    "malignancy",
    "death",
    "muerte",
    "mortalidad",
    "inflammation",
    "unknown",
    "none",
    "n/a",
}

print(f"Overrides de tipo configurados: {len(ENTITY_TYPE_OVERRIDES)}")
print(f"Entidades de ruido configuradas: {len(GENERIC_NOISE_ENTITIES)}")

**5. Etapa 1: Función modular de limpieza estructural (`clean_structural_fields`)**
Preserva columnas crudas de auditoría, limpia marcadores Markdown/AEMPS/ClinicalTrials, unifica guiones Unicode y normaliza espacios.

In [ ]:
def normalize_lookup_key(col_expr):
    """Genera clave de búsqueda en minúsculas, sin tildes y con espacios colapsados."""
    no_accents = F.translate(
        F.lower(F.trim(col_expr)),
        "áéíóúüñàèìòùâêîôû",
        "aeiouunaeiouaeiou",
    )
    return F.trim(F.regexp_replace(no_accents, r"\s+", " "))


def _clean_entity_expr(col_expr):
    """Aplica limpieza determinista de prefijos de prospectos, notas al pie y caracteres Unicode."""
    s = F.trim(col_expr)
    # Eliminar marcadores Markdown de cursiva (_BRCA1_ -> BRCA1)
    s = F.regexp_replace(s, r"_([A-Za-z0-9/\-]+)_", "$1")
    # Eliminar prefijos de frecuencia AEMPS/CIMA y de ClinicalTrials.gov
    s = F.regexp_replace(s, r"(?i)^de forma [^:]+:\s*", "")
    s = F.regexp_replace(s, r"(?i)^procedure:\s*", "")
    # Normalizar flechas de secuencia terapéutica y barras dobles
    s = F.regexp_replace(s, r"\s*▶\s*", " -> ")
    s = F.regexp_replace(s, r"//", "/")
    # Unificar guiones largos Unicode a guion ASCII estándar
    s = F.regexp_replace(s, r"[–—−]", "-")
    # Cerrar corchete abierto en expresiones de TMB-H
    s = F.regexp_replace(s, r"(\[[^\]]+)$", "$1]")
    # Quitar superíndices de nota al pie pegados a palabras largas en minúscula (ej. intersticial2)
    s = F.regexp_replace(s, r"([a-záéíóúñ]{4,})[12+]+$", "$1")
    # Colapsar espacios internos y recortar puntuación residual en extremos
    s = F.regexp_replace(s, r"\s+", " ")
    s = F.regexp_replace(s, r"^[\s.,;:\-\"\'`*]+|[\s.,;:\-\"\'`*]+$", "")
    return F.trim(s)


def clean_structural_fields(df):
    """Preserva valores crudos y estandariza estructura básica de sujeto, predicado, objeto y tipos."""
    return (
        df.withColumn("subject_raw", F.col("sujeto"))
        .withColumn("subject_type_raw", F.col("tipo_sujeto"))
        .withColumn("predicate_raw", F.col("relacion"))
        .withColumn("object_raw", F.col("objeto"))
        .withColumn("object_type_raw", F.col("tipo_objeto"))
        .withColumn("subject", _clean_entity_expr(F.col("sujeto")))
        .withColumn("object", _clean_entity_expr(F.col("objeto")))
        .withColumn("subject_type", F.upper(F.trim(F.regexp_replace(F.col("tipo_sujeto"), r"\s+", "_"))))
        .withColumn("object_type", F.upper(F.trim(F.regexp_replace(F.col("tipo_objeto"), r"\s+", "_"))))
        .withColumn("predicate", F.upper(F.trim(F.regexp_replace(F.col("relacion"), r"\s+", "_"))))
        .withColumn("confidence", F.round(F.col("score").cast("double"), 4))
    )

**6. Etapas 2 y 3: Traducción al inglés y resolución de alias (`resolve_entities_and_translate`)**
Aplica `broadcast` joins contra el diccionario de alias unificando variantes ES/EN, abreviaturas y nombres comerciales sin saturar el codegen de Spark.

In [ ]:
def resolve_entities_and_translate(df, aliases_dict: dict):
    """Resuelve entidades a su nombre canónico en inglés mediante broadcast joins dedicados."""
    alias_schema = T.StructType([
        T.StructField("lookup_key", T.StringType(), False),
        T.StructField("canonical_name", T.StringType(), False),
    ])
    alias_rows = [(k, v) for k, v in aliases_dict.items()]
    alias_base = spark.createDataFrame(alias_rows, schema=alias_schema)

    alias_subj = F.broadcast(
        alias_base.select(
            F.col("lookup_key").alias("_s_lookup"),
            F.col("canonical_name").alias("_s_canon"),
        )
    )
    alias_obj = F.broadcast(
        alias_base.select(
            F.col("lookup_key").alias("_o_lookup"),
            F.col("canonical_name").alias("_o_canon"),
        )
    )

    df_with_keys = (
        df.withColumn("_subj_key", normalize_lookup_key(F.col("subject")))
        .withColumn("_obj_key", normalize_lookup_key(F.col("object")))
    )

    resolved_both = (
        df_with_keys.join(alias_subj, F.col("_subj_key") == F.col("_s_lookup"), "left")
        .join(alias_obj, F.col("_obj_key") == F.col("_o_lookup"), "left")
    )

    # Capitaliza entidades en inglés que venían completamente en minúsculas sin alterar acrónimos mixtos
    def _fallback_case(orig_col):
        return F.when(orig_col == F.lower(orig_col), F.initcap(orig_col)).otherwise(orig_col)

    return (
        resolved_both
        .withColumn("subject", F.coalesce(F.col("_s_canon"), _fallback_case(F.col("subject"))))
        .withColumn("object", F.coalesce(F.col("_o_canon"), _fallback_case(F.col("object"))))
        .withColumn("_subj_key_clean", normalize_lookup_key(F.col("subject")))
        .withColumn("_obj_key_clean", normalize_lookup_key(F.col("object")))
        .drop("_s_lookup", "_s_canon", "_o_lookup", "_o_canon", "_subj_key", "_obj_key")
    )

**7. Etapas 4 y 5: Estandarización de tipos y predicados (`standardize_types_and_predicates`)**
Refina los tipos de entidad (`GENE`, `BIOMARKER`, `PATHWAY`, `SYMPTOM`, `RISK_FACTOR`) y promueve predicados semánticos (`BIOMARKER_FOR`, `HAS_MUTATION`, `RISK_FACTOR_FOR`).

In [ ]:
def standardize_types_and_predicates(df, type_overrides: dict, predicate_map: dict):
    """Aplica overrides de tipo de nodo vía broadcast y normaliza los predicados a UPPER_SNAKE_CASE."""
    override_schema = T.StructType([
        T.StructField("entity_name", T.StringType(), False),
        T.StructField("override_type", T.StringType(), False),
    ])
    override_base = spark.createDataFrame(list(type_overrides.items()), schema=override_schema)

    ov_subj = F.broadcast(
        override_base.select(
            F.col("entity_name").alias("_s_ent"),
            F.col("override_type").alias("_s_ov"),
        )
    )
    ov_obj = F.broadcast(
        override_base.select(
            F.col("entity_name").alias("_o_ent"),
            F.col("override_type").alias("_o_ov"),
        )
    )

    df_types = (
        df.join(ov_subj, F.col("subject") == F.col("_s_ent"), "left")
        .join(ov_obj, F.col("object") == F.col("_o_ent"), "left")
        .withColumn("subject_type", F.coalesce(F.col("_s_ov"), F.col("subject_type")))
        .withColumn("object_type", F.coalesce(F.col("_o_ov"), F.col("object_type")))
        .drop("_s_ent", "_s_ov", "_o_ent", "_o_ov")
    )

    pred_expr = F.create_map([F.lit(item) for kv in predicate_map.items() for item in kv])
    df_pred = df_types.withColumn(
        "predicate",
        F.coalesce(F.element_at(pred_expr, F.col("predicate")), F.col("predicate")),
    )

    # Enriquecimiento semántico determinista de relaciones genéricas ASSOCIATED_WITH
    return df_pred.withColumn(
        "predicate",
        F.when(
            (F.col("predicate") == "ASSOCIATED_WITH")
            & (F.col("subject_type") == "BIOMARKER")
            & (F.col("object_type").isin("DISEASE", "STAGE")),
            F.lit("BIOMARKER_FOR"),
        )
        .when(
            (F.col("predicate") == "ASSOCIATED_WITH")
            & (F.col("subject_type") == "DISEASE")
            & (F.col("object_type").isin("MUTATION", "GENE")),
            F.lit("HAS_MUTATION"),
        )
        .when(
            (F.col("predicate") == "ASSOCIATED_WITH")
            & (F.col("subject_type") == "RISK_FACTOR")
            & (F.col("object_type") == "DISEASE"),
            F.lit("RISK_FACTOR_FOR"),
        )
        .otherwise(F.col("predicate")),
    )

**8. Etapa 6: Validación de reglas de calidad y enrutamiento a cuarentena (`validate_and_split_quarantine`)**
Evalúa nulidad, longitud, confianza, ontología de tipos/predicados, auto-bucles y ruido genérico asignando `_rejection_reason`.

In [ ]:
def validate_and_split_quarantine(
    df,
    valid_node_types: list,
    valid_predicates: set,
    noise_entities: set,
    min_confidence: float = 0.70,
    min_len: int = 2,
    max_len: int = 80,
):
    """Separa las tripletas válidas de las rechazadas etiquetando el motivo exacto de cuarentena."""
    noise_list = sorted(noise_entities)
    is_generic_phase_trial = (
        F.col("_subj_key_clean").rlike(r"^phase [i1]")
        | F.col("_obj_key_clean").rlike(r"^phase [i1]")
    )

    invalid_ontology = (
        ((F.col("predicate") == "COMBINES_WITH") & (F.col("subject_type") == "REGIMEN") & (F.col("object_type") == "REGIMEN"))
        | ((F.col("predicate") == "DIAGNOSES") & (F.col("subject_type") == "DRUG"))
        | ((F.col("predicate") == "TREATS") & (F.col("subject_type").isin("DISEASE", "MUTATION", "GENE", "BIOMARKER", "STAGE", "ADVERSE_EVENT")))
        | ((F.col("predicate") == "EVALUATED_IN") & (F.col("object_type") != "TRIAL"))
        | ((F.col("predicate") == "CAUSES") & (F.col("object_type").isin("DRUG", "REGIMEN", "PROCEDURE", "TRIAL")))
    )

    rejection_reason = (
        F.when(
            F.col("subject").isNull()
            | F.col("object").isNull()
            | F.col("predicate").isNull()
            | (F.length(F.col("subject")) == 0)
            | (F.length(F.col("object")) == 0)
            | (F.length(F.col("predicate")) == 0),
            F.lit("NULL_CORE_FIELD"),
        )
        .when(
            (F.length(F.col("subject")) < min_len)
            | (F.length(F.col("object")) < min_len)
            | (F.length(F.col("subject")) > max_len)
            | (F.length(F.col("object")) > max_len),
            F.lit("INVALID_ENTITY_LENGTH"),
        )
        .when(
            F.col("confidence").isNull()
            | (F.col("confidence") < min_confidence)
            | (F.col("confidence") > 1.0),
            F.lit("LOW_CONFIDENCE_SCORE"),
        )
        .when(~F.col("subject_type").isin(valid_node_types), F.lit("INVALID_SUBJECT_TYPE"))
        .when(~F.col("object_type").isin(valid_node_types), F.lit("INVALID_OBJECT_TYPE"))
        .when(~F.col("predicate").isin(sorted(valid_predicates)), F.lit("INVALID_PREDICATE"))
        .when(F.col("_subj_key_clean") == F.col("_obj_key_clean"), F.lit("SELF_LOOP_TRIPLET"))
        .when(
            F.col("_subj_key_clean").isin(noise_list)
            | F.col("_obj_key_clean").isin(noise_list)
            | is_generic_phase_trial,
            F.lit("GENERIC_OR_NOISE_ENTITY"),
        )
        .when(invalid_ontology, F.lit("INVALID_ONTOLOGY_RELATION"))
        .otherwise(F.lit(None))
    )

    evaluated_df = df.withColumn("_rejection_reason", rejection_reason)
    valid_df = evaluated_df.filter(F.col("_rejection_reason").isNull()).drop("_rejection_reason")
    quarantine_df = (
        evaluated_df.filter(F.col("_rejection_reason").isNotNull())
        .withColumn("_quarantined_at", F.current_timestamp())
    )
    return valid_df, quarantine_df

**9. Etapa 7: Consistencia global de tipos y deduplicación con trazabilidad**
Resuelve cualquier ambigüedad residual de tipos entre documentos y consolida tripletas duplicadas preservando fuentes y conteo de evidencias.

In [ ]:
def resolve_entity_type_consistency(valid_df, type_priority: dict):
    """Unifica variantes residuales de capitalización y asigna un único tipo autoritativo por entidad."""
    prio_expr = F.create_map([F.lit(item) for kv in type_priority.items() for item in kv])

    subj_raw = valid_df.select(F.col("subject").alias("entity"), F.col("subject_type").alias("etype"))
    obj_raw = valid_df.select(F.col("object").alias("entity"), F.col("object_type").alias("etype"))
    all_ents = subj_raw.unionAll(obj_raw)

    # 1. Unificar cualquier variante que difiera únicamente en mayúsculas/minúsculas
    case_counts = (
        all_ents.withColumn("lkey", F.lower(F.trim(F.col("entity"))))
        .groupBy("lkey", "entity")
        .agg(F.count("*").alias("cnt"))
    )
    w_case = Window.partitionBy("lkey").orderBy(F.desc("cnt"), F.asc("entity"))
    canonical_names = (
        case_counts.withColumn("_rn", F.row_number().over(w_case))
        .filter(F.col("_rn") == 1)
        .select("lkey", F.col("entity").alias("canonical_entity"))
    )

    df_cased = (
        valid_df.join(
            F.broadcast(canonical_names.select(F.col("lkey").alias("_sl"), F.col("canonical_entity").alias("_sc"))),
            F.lower(F.trim(F.col("subject"))) == F.col("_sl"),
            "left",
        )
        .join(
            F.broadcast(canonical_names.select(F.col("lkey").alias("_ol"), F.col("canonical_entity").alias("_oc"))),
            F.lower(F.trim(F.col("object"))) == F.col("_ol"),
            "left",
        )
        .withColumn("subject", F.coalesce(F.col("_sc"), F.col("subject")))
        .withColumn("object", F.coalesce(F.col("_oc"), F.col("object")))
        .drop("_sl", "_sc", "_ol", "_oc")
    )

    # 2. Asignar un único tipo autoritativo por frecuencia y prioridad clínica
    subj_ents = df_cased.select(F.col("subject").alias("entity"), F.col("subject_type").alias("etype"))
    obj_ents = df_cased.select(F.col("object").alias("entity"), F.col("object_type").alias("etype"))

    counts_df = subj_ents.unionAll(obj_ents).groupBy("entity", "etype").agg(F.count("*").alias("cnt"))
    w = Window.partitionBy("entity").orderBy(
        F.desc("cnt"),
        F.coalesce(F.element_at(prio_expr, F.col("etype")), F.lit(99)),
        F.asc("etype"),
    )
    canonical_types = (
        counts_df.withColumn("_rn", F.row_number().over(w))
        .filter(F.col("_rn") == 1)
        .select(F.col("entity"), F.col("etype").alias("authoritative_type"))
    )

    return (
        df_cased.join(
            F.broadcast(canonical_types.select(F.col("entity").alias("_se"), F.col("authoritative_type").alias("_st"))),
            F.col("subject") == F.col("_se"),
            "left",
        )
        .join(
            F.broadcast(canonical_types.select(F.col("entity").alias("_oe"), F.col("authoritative_type").alias("_ot"))),
            F.col("object") == F.col("_oe"),
            "left",
        )
        .withColumn("subject_type", F.coalesce(F.col("_st"), F.col("subject_type")))
        .withColumn("object_type", F.coalesce(F.col("_ot"), F.col("object_type")))
        .drop("_se", "_st", "_oe", "_ot")
    )


def deduplicate_triplets(consistent_df):
    """Consolida tripletas repetidas agregando evidencias, confianza y trazabilidad de origen."""
    return (
        consistent_df.groupBy("subject", "subject_type", "predicate", "object", "object_type")
        .agg(
            F.count("*").cast("int").alias("evidence_count"),
            F.round(F.avg("confidence"), 4).alias("avg_confidence"),
            F.round(F.max("confidence"), 4).alias("max_confidence"),
            F.countDistinct("doc_id").cast("int").alias("doc_count"),
            F.array_sort(F.array_distinct(F.collect_list("doc_id"))).alias("source_docs"),
            F.array_sort(F.array_distinct(F.collect_list("_source_file"))).alias("source_files"),
            F.array_sort(F.array_distinct(F.collect_list("_source_folder"))).alias("source_folders"),
            F.array_sort(F.array_distinct(F.collect_list("_source_domain"))).alias("source_domains"),
        )
        .withColumn("_processed_at", F.current_timestamp())
    )

**10. Ejecución del pipeline Silver y persistencia en Delta Lake**
Orquesta las 7 etapas sobre `bronze.pdac.raw_triplets` y escribe las tablas `clean_triplets` y `quarantine_triplets`.

In [ ]:
bronze_raw_df = spark.table(SOURCE_TABLE)

df_step1 = clean_structural_fields(bronze_raw_df)
df_step2 = resolve_entities_and_translate(df_step1, ENTITY_ALIASES)
df_step3 = standardize_types_and_predicates(df_step2, ENTITY_TYPE_OVERRIDES, PREDICATE_MAP)

valid_pre_dedup_df, quarantine_df = validate_and_split_quarantine(
    df_step3,
    valid_node_types=VALID_NODE_TYPES,
    valid_predicates=VALID_PREDICATES,
    noise_entities=GENERIC_NOISE_ENTITIES,
    min_confidence=MIN_CONFIDENCE,
    min_len=MIN_ENTITY_LEN,
    max_len=MAX_ENTITY_LEN,
)

consistent_valid_df = resolve_entity_type_consistency(valid_pre_dedup_df, NODE_TYPE_PRIORITY)
clean_triplets_df = deduplicate_triplets(consistent_valid_df)

(
    clean_triplets_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(CLEAN_TABLE)
)

quarantine_out_df = quarantine_df.drop("_subj_key_clean", "_obj_key_clean")
(
    quarantine_out_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(QUARANTINE_TABLE)
)

spark.sql(f"COMMENT ON TABLE {CLEAN_TABLE} IS 'Capa Silver: Tripletas biomédicas limpias, traducidas al inglés, resueltas y deduplicadas.'")
spark.sql(f"COMMENT ON TABLE {QUARANTINE_TABLE} IS 'Capa Silver: Tripletas rechazadas con motivo explícito de cuarentena (_rejection_reason).'")

print(f"Escrita tabla limpia     : {CLEAN_TABLE}")
print(f"Escrita tabla cuarentena : {QUARANTINE_TABLE}")

**11. Etapa 8: Auditoría y métricas de calidad Antes vs. Después**
Calcula la reducción de duplicados y entidades tras alias, desglosa la cuarentena por motivo y guarda `silver.pdac.quality_metrics`.

In [ ]:
bronze_audit_df = spark.table(SOURCE_TABLE)
clean_audit_df = spark.table(CLEAN_TABLE)
quarantine_audit_df = spark.table(QUARANTINE_TABLE)

raw_total = bronze_audit_df.count()
clean_unique_count = clean_audit_df.count()
valid_pre_dedup_count = (
    clean_audit_df.agg(F.coalesce(F.sum("evidence_count"), F.lit(0))).collect()[0][0]
)
quarantine_count = quarantine_audit_df.count()

raw_unique_cs = (
    bronze_audit_df.select(F.col("sujeto").alias("e"))
    .unionAll(bronze_audit_df.select(F.col("objeto").alias("e")))
    .filter(F.col("e").isNotNull())
    .distinct()
    .count()
)
raw_unique_ci = (
    bronze_audit_df.select(F.lower(F.trim(F.col("sujeto"))).alias("e"))
    .unionAll(bronze_audit_df.select(F.lower(F.trim(F.col("objeto"))).alias("e")))
    .filter(F.col("e").isNotNull())
    .distinct()
    .count()
)
clean_unique_nodes = (
    clean_audit_df.select(F.col("subject").alias("e"))
    .unionAll(clean_audit_df.select(F.col("object").alias("e")))
    .distinct()
    .count()
)

node_reduction_pct = round((1.0 - clean_unique_nodes / max(raw_unique_cs, 1)) * 100, 2)
dedup_reduction_pct = round((1.0 - clean_unique_count / max(valid_pre_dedup_count, 1)) * 100, 2)
quarantine_pct = round((quarantine_count / max(raw_total, 1)) * 100, 2)

summary_rows = [
    ("01_bronze_raw_triplets", int(raw_total), "Total tripletas ingeridas en Bronze"),
    ("02_silver_valid_pre_dedup", int(valid_pre_dedup_count), "Tripletas válidas antes de deduplicar"),
    ("03_silver_quarantined_triplets", int(quarantine_count), f"Tripletas en cuarentena ({quarantine_pct}%)"),
    ("04_silver_clean_unique_triplets", int(clean_unique_count), f"Tripletas únicas en Silver (-{dedup_reduction_pct}% duplicados)"),
    ("05_raw_unique_entities_case_sensitive", int(raw_unique_cs), "Entidades crudas únicas (sensible a mayúsculas)"),
    ("06_raw_unique_entities_case_insensitive", int(raw_unique_ci), "Entidades crudas únicas (insensible a mayúsculas)"),
    ("07_silver_unique_canonical_entities", int(clean_unique_nodes), f"Nodos canónicos en inglés (-{node_reduction_pct}% vs crudo)"),
]

metrics_df = (
    spark.createDataFrame(summary_rows, ["metric_name", "metric_value", "notes"])
    .withColumn("_audited_at", F.current_timestamp())
)
metrics_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(METRICS_TABLE)

print("=== RESUMEN DE CALIDAD (ANTES VS. DESPUÉS) ===")
metrics_df.select("metric_name", "metric_value", "notes").show(truncate=False)

print("=== DESGLOSE DE CUARENTENA POR MOTIVO ===")
quarantine_audit_df.groupBy("_rejection_reason").count().orderBy(F.desc("count")).show(truncate=False)

print("=== DISTRIBUCIÓN DE ENTIDADES CANÓNICAS POR TIPO ===")
(
    clean_audit_df.select(F.col("subject").alias("entity"), F.col("subject_type").alias("node_type"))
    .union(clean_audit_df.select(F.col("object").alias("entity"), F.col("object_type").alias("node_type")))
    .distinct()
    .groupBy("node_type")
    .count()
    .orderBy(F.desc("count"))
    .show(truncate=False)
)

print("=== DISTRIBUCIÓN DE RELACIONES LIMPIAS POR PREDICADO ===")
clean_audit_df.groupBy("predicate").agg(
    F.count("*").alias("unique_edges"),
    F.sum("evidence_count").alias("total_mentions"),
).orderBy(F.desc("unique_edges")).show(truncate=False)

display(clean_audit_df.orderBy(F.desc("evidence_count"), F.desc("avg_confidence")).limit(15))